<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/proteomics-multiomics/lessons/P07-L04-factor-models-two-omics/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/proteomics-multiomics/lessons/P07-L04-factor-models-two-omics/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/proteomics-multiomics/lessons/P07-L04-factor-models-two-omics/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/proteomics-multiomics/lessons/P07-L04-factor-models-two-omics/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P07-L04 · Factor models for two omics layers: a MOFA-style model built from scratch and checked against planted biology

**You will build:** the step of a multi-omics pipeline that explains two matched molecular layers (a
proteomics-style one and a transcriptomics-style one, measured on the same donors) with a small set of
donor-level **factors**, each with a weight for every feature in each layer, and that says which factors
live in which layer. You will write the data preparation, a concatenated-PCA baseline, a matching of
recovered factors to planted ones, a variance-explained table, and the model itself: alternating ridge
regressions with a scale for every factor in every layer, which switches a factor off where it has no
work to do. Then you will **measure**, against an answer key, what the model recovers that PCA does not,
how stable it is, and where it stops helping.

**Time:** ~150 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download
· **Prerequisites:** `P07-L02-imputation-and-batch-effects` (a donor-by-feature matrix on a log scale, a
technical batch, and why a number from one simulated world is one draw) and `T01-L01-statistics-by-simulation`
(a result is a distribution over seeds, not a single number). Python and numpy, including matrix products
and `np.linalg.solve`. No biology beyond "each layer is a table of numbers per donor".

By the end you will be able to:

1. Implement view preparation, a concatenated-PCA baseline, an optimal one-to-one matching of recovered to
   planted factors and a per-layer variance-explained table, so that any set of factors can be scored
   against an answer key.
2. Implement a multi-view factor model as alternating regularised least squares (a ridge update for each
   layer's weights, a noise-weighted ridge update for the shared donor factors, and a scale for every
   factor in every layer) and check that its objective never rises.
3. Measure, against planted factors, how well concatenated PCA and the factor model recover shared,
   layer-specific and technical factors: by correlation, by variance explained in each layer, and by how
   many factors each one reports.
4. Measure how stable the recovered factors are across random restarts, across the number of sweeps and
   across simulated worlds, and choose a restart without using the answer key.
5. Explain, from the printed numbers, when concatenated PCA and the factor model agree, when a tie between
   two factors leaves PCA's components arbitrary, and what this lesson's model leaves out (the inference
   MOFA+ uses, per-feature noise, and deep generative models such as totalVI).

**The data is synthetic, and the simulator is in this notebook.** Real data has no answer key: nobody knows
which factors are "really there". A simulated cohort does, so "check recovered factors against known biology"
can only mean "against known generating factors". `meta.yaml` declares the data as synthetic and nothing is
downloaded. The factor names are labels for a planted structure, not claims about biology, and nothing you
conclude about these counts describes a real cohort or a real software package. What transfers is the
method, and the habit of scoring a factor model against something it could have got wrong.

**What this lesson does not do.** The catalogue's lab also runs totalVI on CITE-seq data. As of 14 October 2026,
totalVI's package, scvi-tools, depends on PyTorch, and totalVI's documentation lists among its limitations:
"Effectively requires a GPU for fast inference." MOFA+'s own Python package needs a software stack that is not
in this course's environment either (`claims.yaml`). So that half is left out, and what you build is the
factor-model half, from first principles, at a size a laptop can run.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `P07-L02`, `T01-L01`

**This lesson assumes you can already:**

- multiply and transpose matrices in numpy, and solve a small linear system with np.linalg.solve
- say what the singular value decomposition of a data matrix returns, and what a principal component is
- fit a ridge regression, which is least squares with a penalty on the size of the weights
- compute a Pearson correlation between two columns of numbers, and say why its sign alone means little when a column can be flipped

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [MOFA2 documentation: tutorials and FAQ](https://biofam.github.io/MOFA2/) · bioFAM (the MOFA authors) · *free* — its FAQ and tutorials show how the factors, weights and variance-explained plots of a real MOFA model are read, which this lesson reproduces on a planted cohort
- [MOFA+: a statistical framework for comprehensive integration of multi-modal single-cell data](https://pmc.ncbi.nlm.nih.gov/articles/PMC7212577/) · Argelaguet et al. (Genome Biology 2020, open access, PubMed Central) · *free* — its Methods define the model the lesson simplifies, the variance decomposition that exercise 4 imitates (with refitted weights), and the model selection and PCA start of sections 13 and 16
- [Multi-Omics Factor Analysis - a framework for unsupervised integration of multi-omics data sets](https://pmc.ncbi.nlm.nih.gov/articles/PMC6010767/) · Argelaguet et al. (Molecular Systems Biology 2018, open access, PubMed Central) · *free* — the original model, with the view-wise and factor-wise sparsity prior that the lesson's scales imitate, and the random restarts that section 16 measures
- [The Elements of Statistical Learning, 2nd edition](https://hastie.su.domains/ElemStatLearn/) · Trevor Hastie, Robert Tibshirani and Jerome Friedman (Springer; PDF free from the authors) · *free* — section 3.4 on ridge regression, which is what every update in the model is, and section 14.5 on principal components, the baseline
- [numpy.linalg.solve](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html) · NumPy documentation · *free* — the call that exercises 5 and 6 are built on, with the shapes it takes and returns when the right-hand side is a matrix

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import itertools
import math
import os
import signal
import sys
import threading
import time
import traceback
from typing import Callable, NamedTuple

import numpy as np

import matplotlib
_INTERACTIVE = "ipykernel" in sys.modules
if not _INTERACTIVE:
    # Headless: a script run (including this repository's execution gate) must never try to
    # open a window. In Jupyter the default inline backend is already the right one.
    matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402  (backend must be chosen before this import)

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__,
      "· matplotlib", matplotlib.__version__)

# The world. Every number that shapes it is here, so that nothing later is a hidden knob.
SEED = 20261014          # the world you study in detail; later sections also use seeds after it
N_WORLDS = 8             # worlds in the seed-spread section
N_DONORS = 150
N_PROTEIN, N_RNA = 200, 600          # features in the proteomics-style and the transcriptomics-style layer
FACTOR_NAMES = ("inflammation", "proliferation", "stress response", "secretion", "batch")
VIEW_NAMES = ("protein", "RNA")
# The share of each layer's variance that each planted factor carries. Stress response is present only in
# the RNA layer, and its share is set by the tie rule in generate_cohort (so the 0.0 in that slot is not used).
SHARE_PROTEIN = (0.16, 0.08, 0.0, 0.07, 0.05)
SHARE_RNA = (0.08, 0.07, 0.0, 0.0, 0.05)
DENSITY = (0.5, 0.4, 0.3, 0.25, 0.6)       # the fraction of a layer's features a factor touches
RAW_SD = (0.8, 1.5)                         # the raw per-entry standard deviation of each layer
RAW_MEAN = (24.0, 3.0)                      # and its typical level (log2 intensity-like; log expression-like)
BATCH_FACTOR = 4                            # which planted factor is the technical one

# The analysis settings, fixed for the whole lesson so that only the choices you study vary.
K_FIT = 8                # factors the model starts with: more than the world has, so it must switch some off
N_SWEEPS = 1000          # sweeps of the alternating updates
ARD_FROM = 30            # sweeps with every scale held at 1 before the scales start to adapt
FLOOR = 1e-6             # the smallest scale a factor can have in a layer
ACTIVE_R2 = 0.03         # a factor is "active" in a layer when it explains at least this share of it
N_RESTARTS = 8
GAPS = (1.0, 1.5, 2.0)          # strength of proliferation over strength of stress response
WORLDS_PER_GAP = 4              # the same worlds (the same seeds) at every gap
WEAK_SHARES = (0.6, 0.45)       # section 19: multiples of the planted shares of variance
WEAK_DONORS = 100               # section 19: donors in the weaker worlds
WEAK_WORLDS = 5


def _show(fig: "matplotlib.figure.Figure") -> None:
    """Display a figure in Jupyter, or close it cleanly in a headless script run."""
    if _INTERACTIVE:
        plt.show()
    else:
        plt.close(fig)


_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("prepare_views",),
    "exercise 2": ("concatenated_pca",),
    "exercise 3": ("best_assignment",),
    "exercise 4": ("variance_explained",),
    "exercise 5": ("update_weights",),
    "exercise 6": ("update_factors",),
    "exercise 7": ("update_scales",),
    "exercise 8": ("fit_factor_model",),
    "exercise 9": ("consensus_run",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run
_UPTO = {i: tuple(f"exercise {k}" for k in range(1, i + 1)) for i in range(1, 10)}
# The heavy demos (a whole cohort, then many) run when the file is run as a notebook or a script, and are
# skipped when the autograder imports it, so grading stays quick.
_RUN_DEMOS = __name__ == "__main__"


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (best_assignment)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the message. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#  * It never raises inside its own set-up, exit or clean-up, and on the way out it takes any
#    alarm of its own that has gone off but not yet been handled (with SIGALRM blocked) before
#    the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
# How long an exercise's check may run before it counts as stuck, and how long any other cell that runs your
# code (a demonstration, a table) may: a right answer needs a small fraction of either, measured on the worked
# solution (internal/retrofit_guard.py, 10x or more). tools/verify_all.py reads both numbers.
_CHECK_SECONDS = 2
_DEMO_SECONDS = 45


class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s in case a bare
    `except:` does. `_time_limit` turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 2048):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever, or work grows far faster than the number of factors (every way of pairing two long lists)")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = (),
         limit: float = 0) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    An exercise check runs under a limit of `_CHECK_SECONDS` (2 s; the reference's slowest check takes a few
    milliseconds) and any other cell under `_DEMO_SECONDS` (45 s), and the guard also stops a check whose memory
    keeps growing, so a loop that never ends, or one that adds to a list for ever, fails with a message instead of
    freezing the notebook or filling its memory. A stub you have not filled in yet simply says so. A wrong answer
    prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        with _time_limit(limit or (_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS), label):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def _demo(label: str, fn: Callable[[], None], needs: tuple[str, ...] = (), limit: int = 0) -> None:
    """A demo cell: guarded like a check, and skipped when the file is imported by the grader."""
    if _RUN_DEMOS:
        _try(label, fn, needs, limit)


_PIPE: dict = {}    # fits and tables shared between the demo cells, so no cell recomputes another's work

## 1. The problem in one page

A cohort of donors has been measured twice: a proteomics-style layer (one number per protein per donor) and a
transcriptomics-style layer (one number per gene per donor). A **factor model** says that both layers are
driven by a handful of hidden donor-level scores, the **factors**: layer `m` is about `Z @ W_m.T`, where `Z`
has one row per donor and one column per factor, and `W_m` has one row per feature and one column per
factor. The weights say which features follow which factor, in which layer. This is the model
family MOFA belongs to: it discovers sources of variation that are shared across layers and ones that are
specific to a single layer (`claims.yaml`).

Three kinds of factor matter, and the lesson plants each kind in a simulated cohort, so that you can
check what you recover against what you put in:

- **shared** factors, which move features in both layers;
- **layer-specific** factors, which move features in one layer only, and which a single table of the
  layers side by side does not label for you;
- a **technical** factor (a batch), which is not biology but is in the data, and which you want to find
  rather than mistake for biology.

Three things make it harder than it looks, and each has its own section:

- **Rotation.** The reconstruction `Z @ W.T` does not change if you mix the factors with an invertible matrix
  and the weights with its inverse, and a model whose factors are independent and standard normal is left
  unchanged by any orthogonal mix (`claims.yaml`). The solution is therefore not unique, and "the recovered
  factor" has no meaning until something decides which mixture to report. PCA decides by variance; the factor
  model you build decides by a per-layer scale that rewards a factor for being absent from a layer.
- **How many factors.** Nobody tells you. You start with more than you need and let the model switch the
  surplus off.
- **Restarts.** The objective has more than one good-looking answer, and the answer you get depends on where
  the iteration starts. The MOFA papers address this with random restarts scored by the ELBO, a bound
  on the evidence of the whole model (section 16), and MOFA+ later switched to starting from principal components
  (`claims.yaml`).

The plan. Section 2 builds the cohort. Sections 3 to 7 write the baseline and the scoring tools and measure the
baseline. Sections 8 to 13 write the model and run it on the cohort. Sections 14 to 19 measure it: with its key
ingredient removed, across sweeps, across restarts, across a sweep from a tie to well-separated factors, across
several worlds, and with the planted factors made weaker.

In [ ]:
# A toy before the real cohort: four donors, ONE factor, two layers, no noise. Each layer is the factor times a column of
# weights, so each layer is a table whose every column is a multiple of the same four donor scores.
toy_z = np.array([[-1.0], [0.0], [1.0], [2.0]])
toy_w = [np.array([[2.0], [-1.0], [0.0]]), np.array([[0.5], [1.0]])]
for toy_name, toy_weights in zip(VIEW_NAMES, toy_w):
    print(f"{toy_name} layer = Z @ W.T (one row per donor, one column per feature):")
    print(np.round(toy_z @ toy_weights.T, 2))

## 2. The cohort you will study

`generate_cohort` (given, below) simulates donors, two layers, the planted factors and the answer key.
Read it before you analyse anything; the point of a synthetic cohort is that you are allowed to know exactly
how it was made.

- Every factor is a column of donor scores. All are standard normal except `batch`, the technical one: a 0/1
  label for which batch each donor was measured in, standardised.
- A factor touches a fraction of each layer's features, with random weights scaled so that the factor
  carries a chosen **share of that layer's variance** (the constants `SHARE_PROTEIN` and `SHARE_RNA`). A
  share of zero means the factor is absent from that layer: inflammation and proliferation are shared,
  stress response is RNA-only, secretion is protein-only, and batch is in both.
- **The tie.** A factor's total strength is the sum, over layers, of the number of features times its share of
  that layer. Proliferation and stress response are given the *same* total strength (divided by `gap`,
  which is 1 here), so that nothing in the variance of the data tells them apart.
- The layers are on different raw scales (a different level, a different spread, a different number of
  features), as real layers are, so the preparation in section 3 has something to do.
- **The strengths were set with the model in mind.** Every planted share is clearly above the activity
  threshold `ACTIVE_R2` used later, and there are enough donors for each factor to be estimated. Section 19
  weakens the factors (the `share_scale` argument) to find where the model you build stops working, so do
  not read sections 13 to 18 as a property of the method alone.

In [ ]:
class Cohort(NamedTuple):
    """A simulated cohort, and the answer key that generated it."""
    views: list            # [protein layer, RNA layer]: (donors, features) float arrays on a raw scale
    factors: np.ndarray    # (donors, 5) the planted factor values: the answer key
    weights: list          # the planted weights, [(200, 5), (600, 5)]
    batch: np.ndarray      # (donors,) 0 or 1: the technical batch of each donor
    gap: float             # strength of proliferation over strength of stress response


def generate_cohort(seed: int, gap: float = 1.0, n_donors: int = N_DONORS, share_scale: float = 1.0) -> Cohort:
    """Given to you. Simulate matched protein-like and RNA-like layers from the planted factors.

    Only `standard_normal` and `random` are drawn from the generator.
    `gap` is the strength of proliferation divided by the strength of stress response: 1.0 is a tie, and
    larger values make stress response weaker. Strength is the sum over layers of (features x share of
    that layer's variance), which is what concatenated PCA sees once the layers are on one scale.
    `share_scale` multiplies every planted share of variance (1.0 is the world studied in sections 2 to 18);
    smaller values make every factor weaker and the noise a larger part of each layer.
    """
    rng = np.random.default_rng(seed)
    n_factors = len(FACTOR_NAMES)
    z = rng.standard_normal((n_donors, n_factors))
    in_batch = np.zeros(n_donors, dtype=int)
    in_batch[np.argsort(rng.random(n_donors), kind="stable")[: n_donors // 2]] = 1
    z[:, :BATCH_FACTOR] = (z[:, :BATCH_FACTOR] - z[:, :BATCH_FACTOR].mean(axis=0)) / z[:, :BATCH_FACTOR].std(axis=0)
    z[:, BATCH_FACTOR] = (in_batch - in_batch.mean()) / in_batch.std()
    sizes = (N_PROTEIN, N_RNA)
    shares = [share_scale * np.array(SHARE_PROTEIN, dtype=float), share_scale * np.array(SHARE_RNA, dtype=float)]
    strength = sum(sizes[m] * shares[m][1] for m in (0, 1))                 # proliferation, all layers
    shares[1][2] = strength / gap / sizes[1]                                # the tie rule
    views, weights = [], []
    for m, p in enumerate(sizes):
        w = np.zeros((p, n_factors))
        for k in range(n_factors):
            if shares[m][k] == 0.0:
                continue
            count = max(3, int(round(DENSITY[k] * p)))
            touched = np.argsort(rng.random(p), kind="stable")[:count]
            values = rng.standard_normal(count)
            w[touched, k] = values * math.sqrt(shares[m][k] * RAW_SD[m] ** 2 * p / float(values @ values))
        noise_sd = RAW_SD[m] * math.sqrt(1.0 - shares[m].sum())
        noise = noise_sd * rng.standard_normal((n_donors, p))
        level = RAW_MEAN[m] + rng.standard_normal(p)
        views.append(z @ w.T + noise + level)
        weights.append(w)
    return Cohort(views, z, weights, in_batch, float(gap))


def init_factors(views: list, k: int, seed: int | None = None) -> np.ndarray:
    """Given to you. A starting point for the factors: random (an integer `seed`) or from PCA (`seed=None`).

    The PCA start is the first `k` principal components of the layers side by side, scaled to unit
    variance; the random start is standard normal numbers. `views` should already be prepared.
    """
    n = views[0].shape[0]
    if seed is None:
        u, _, _ = np.linalg.svd(np.hstack(views), full_matrices=False)
        return u[:, :k] * math.sqrt(n)
    return np.random.default_rng(seed).standard_normal((n, k))


def abs_corr_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Given to you. The absolute Pearson correlation between every column of `a` and every column of `b`.

    Absolute, because the sign of a factor is arbitrary: flipping a factor and its weights changes nothing.
    A column with no variation (sum of squared deviations at or below 1e-12) has correlation 0 with everything.
    """
    a = a - a.mean(axis=0)
    b = b - b.mean(axis=0)
    na = np.sqrt((a ** 2).sum(axis=0))
    nb = np.sqrt((b ** 2).sum(axis=0))
    na = np.where(na ** 2 <= 1e-12, np.inf, na)
    nb = np.where(nb ** 2 <= 1e-12, np.inf, nb)
    return np.abs((a / na).T @ (b / nb))


def span_r2(fitted: np.ndarray, planted: np.ndarray) -> np.ndarray:
    """Given to you. How much of each planted factor lies in the span of ALL the fitted factors together.

    The share of each planted column's variance that a least-squares regression on every fitted column
    explains. Unlike a correlation with one fitted column, it does not care how the fitted factors are mixed.
    """
    f = fitted - fitted.mean(axis=0)
    p = planted - planted.mean(axis=0)
    coef, *_ = np.linalg.lstsq(f, p, rcond=None)
    return 1.0 - ((p - f @ coef) ** 2).sum(axis=0) / (p ** 2).sum(axis=0)


def objective(views: list, z: np.ndarray, weights: list, noise_var, scale2) -> float:
    """Given to you. The quantity every update in this lesson lowers, up to additive constants.

    With `n` donors, `p_m` features in layer `m`, factors `z` (n, K), weights `W_m` (p_m, K), one noise
    variance `s_m` per layer and one scale `a_mk` per layer and factor:

        J = 1/2 ||Z||^2 + sum_m [ n p_m / 2 log s_m + ||Y_m - Z W_m^T||^2 / (2 s_m)
                                   + sum_k ( p_m / 2 log a_mk + ||w_mk||^2 / (2 a_mk) ) ]

    The factors have a unit-variance prior, the weights of factor k in layer m have a prior variance
    `a_mk`, and the data have noise variance `s_m`. The last bracket is what lets a factor be switched off
    in one layer: each weight column pays `p_m / 2 log a_mk`, which falls without limit as `a_mk` falls.
    That is also why `J` has no lowest value: with a factor's weights at 0 in a layer, `J` keeps falling as
    its scale does, and only the floor `FLOOR` that the updates apply to every scale stops it.
    """
    n = z.shape[0]
    total = 0.5 * float((z ** 2).sum())
    for m, y in enumerate(views):
        p = y.shape[1]
        resid = y - z @ weights[m].T
        total += 0.5 * n * p * math.log(noise_var[m]) + float((resid ** 2).sum()) / (2.0 * noise_var[m])
        total += float((0.5 * p * np.log(scale2[m]) + (weights[m] ** 2).sum(axis=0) / (2.0 * scale2[m])).sum())
    return total


def data_nll(views: list, weights: list, noise_var) -> float:
    """Given to you. Minus the log-likelihood of the data under the fitted factor model, factors integrated out.

    Each donor's features (both layers side by side) are normal with covariance `W W^T + D`, where `D` holds
    the noise variance of each layer. Computed with the identities for a low-rank-plus-diagonal matrix, so
    no features-by-features matrix is ever built. Constants are dropped. Unlike `objective`, it contains no
    prior on the weights, so it cannot see which way the factors are mixed.
    """
    n, k = views[0].shape[0], weights[0].shape[1]
    inner, cross = np.eye(k), np.zeros((n, k))
    logdet = quad = 0.0
    for y, w, s in zip(views, weights, noise_var):
        inner = inner + w.T @ w / s
        cross = cross + y @ w / s
        logdet += y.shape[1] * math.log(s)
        quad += float((y ** 2).sum()) / s
    logdet += float(np.linalg.slogdet(inner)[1])
    quad -= float(np.trace(np.linalg.solve(inner, cross.T @ cross)))
    return 0.5 * (n * logdet + quad)


def _given_check_nll() -> None:
    """The given likelihood is checked against the dense formula it avoids, on a small cohort."""
    c = generate_cohort(3, n_donors=12)
    views = [c.views[0][:, :6] - c.views[0][:, :6].mean(axis=0), c.views[1][:, :5] - c.views[1][:, :5].mean(axis=0)]
    rng = np.random.default_rng(1)
    w = [rng.standard_normal((6, 2)), rng.standard_normal((5, 2))]
    nv = [0.7, 1.3]
    cov = np.vstack(w) @ np.vstack(w).T + np.diag([0.7] * 6 + [1.3] * 5)
    x = np.hstack(views)
    dense = 0.5 * (12 * np.linalg.slogdet(cov)[1] + np.trace(np.linalg.solve(cov, x.T @ x)))
    assert abs(data_nll(views, w, nv) - dense) < 1e-8 * abs(dense), "the given data_nll disagrees with the dense formula"
    print("the given likelihood agrees with the dense formula on a small cohort")


_try("given: likelihood", _given_check_nll)

In [ ]:
COHORT = generate_cohort(SEED)


def _describe_cohort() -> None:
    c = COHORT
    n = c.factors.shape[0]
    print(f"cohort (seed {SEED}): {n} donors, {int(c.batch.sum())} in batch 1 and {n - int(c.batch.sum())} in batch 0")
    totals = []
    for m, y in enumerate(c.views):
        yc = y - y.mean(axis=0)
        totals.append(float((yc ** 2).sum()))
        print(f"  {VIEW_NAMES[m]:8s} layer: {y.shape[1]} features, typical level {float(y.mean()):5.1f}, "
              f"spread of a feature across donors {float(np.sqrt((yc ** 2).mean())):.2f} (raw units)")
    print(f"  the RNA layer carries {totals[1] / totals[0]:.1f} times the total variance of the protein layer "
          "before any preparation")
    print("  planted factors, and the number of features each one touches:")
    print(f"    {'factor':16s} {'in protein':>10s} {'in RNA':>8s}")
    for k, name in enumerate(FACTOR_NAMES):
        touched = [int((w[:, k] != 0).sum()) for w in c.weights]
        print(f"    {name:16s} {touched[0]:10d} {touched[1]:8d}")
    strength = [sum(float((w[:, k] ** 2).sum()) / RAW_SD[m] ** 2 for m, w in enumerate(c.weights))
                for k in range(len(FACTOR_NAMES))]
    print("  planted strength (sum over layers of features x share of the layer's variance):")
    print("    " + ", ".join(f"{FACTOR_NAMES[k]} {strength[k]:.1f}" for k in range(len(FACTOR_NAMES))))


_demo("the cohort", _describe_cohort)

Read the table before you move on. Which factors are absent from which layer? Which two have the same
strength? And what does the line about the RNA layer's total variance say about what PCA would do with
these layers if you concatenated them as they stand? You will measure that in section 7, not assume it.

## 3. Exercise 1 — `prepare_views()`

A factor model sees numbers, not units. Two things must hold before you compare layers, or pool them.

- **Every feature is centred.** The model has no intercept: a protein's average level is not information
  about donors, and left in, it becomes the strongest "factor" in the data.
- **Every layer is on one scale.** A layer with more features, or a larger spread, carries more total
  variance, and anything that pools the layers is steered by it. The MOFA documentation says it plainly:
  bigger data modalities tend to be over-represented in the factors (`claims.yaml`).

The rule here: subtract each feature's mean over donors, then divide the **whole layer** by the square root
of the mean of its squared entries, so that every prepared layer has a mean squared entry of exactly 1.
That equalises the layers' total variance *per entry*; a layer with more features still has more entries,
and the factor model will weigh the layers by its own noise estimates, which is the next step.

<details><summary>💡 Hint 1 — what to think about</summary>

Two means are involved and they are different: one per feature (over donors), and one over every entry of
the whole layer (after centring). Which one do you subtract, and which one do you divide by?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

For each layer: subtract its column means, take the mean of the squares of everything that is left, divide
the layer by the square root of that. Raise `ValueError` before doing anything else for an empty list, a
layer that is not two-dimensional, layers with different numbers of donors, fewer than two donors, a value
that is not finite, or a layer with no variation. "No variation" means a mean square after centring every
feature on its own mean of at most `2.0 ** -66` (about 1.4e-20; a power of two, so that the edge is exact in
binary): a constant column whose mean is not exact in binary still leaves a tiny remainder, and a layer
whose columns are each constant, at different levels, has no variation either.
</details>

In [ ]:
def prepare_views(views: list) -> list:
    """Centre every feature of every layer, then put every layer on one scale.

    For each layer (donors x features): subtract each feature's mean over donors, then divide the whole
    layer by the square root of the mean of its squared entries. The result has a mean squared entry of
    1. New arrays are returned; the inputs are not changed.

    Raise `ValueError` for: an empty list; a layer that is not 2-D; layers whose numbers of donors
    differ; fewer than 2 donors; any value that is not finite; a layer whose centred mean square (every feature
    centred on its own mean) is at most `2.0 ** -66` (about 1.4e-20; no variation).

    Example:
        >>> a = np.array([[1.0, 10.0], [3.0, 14.0]])
        >>> prepare_views([a])[0].round(4).tolist()
        [[-0.6325, -1.2649], [0.6325, 1.2649]]

    Returns:
        A list of float arrays with the shapes of the inputs.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_prepare() -> None:
    a = np.array([[1.0, 10.0], [3.0, 14.0]])
    got = prepare_views([a])[0]
    assert np.allclose(got, np.array([[-1.0, -2.0], [1.0, 2.0]]) / math.sqrt(2.5), rtol=0, atol=1e-12), (
        "centre each FEATURE (column) first, then divide the whole layer by the root of its mean square: "
        f"expected [[-0.6325, -1.2649], [0.6325, 1.2649]], got {np.round(got, 4).tolist()}")
    b = np.array([[0.0, 5.0, 1.0], [2.0, 5.0, -1.0], [4.0, 5.0, 3.0]])
    c = np.array([[2.0], [4.0], [9.0]])
    before = b.copy()
    out = prepare_views([b, c])
    assert np.array_equal(b, before), "do not change the input in place"
    assert len(out) == 2 and out[0].shape == b.shape and out[1].shape == c.shape, "one prepared layer per layer, same shapes"
    for layer in out:
        assert abs(float((layer ** 2).mean()) - 1.0) < 1e-12, "every prepared layer must have a mean squared entry of 1"
        assert np.allclose(layer.mean(axis=0), 0.0, atol=1e-12), "every feature must have mean 0 after preparation"
    assert np.allclose(out[0][:, 1], 0.0), "a constant feature stays at 0: it is centred, and the layer is scaled as a whole"
    for bad in ([], [np.ones(3)], [np.ones((3, 2)), np.ones((4, 2))], [np.ones((1, 2))],
                [np.array([[1.0, np.nan], [2.0, 3.0]])], [np.full((4, 2), 21.3)],
                [np.array([[1.0, 5.0], [1.0, 5.0], [1.0, 5.0]])]):
        try:
            prepare_views(bad)
        except ValueError:
            continue
        raise AssertionError("an empty list, a 1-D layer, layers with different donors, one donor, a NaN, or a layer with "
                             "no variation must raise ValueError (a layer whose features are each constant, at different "
                             "levels, has no variation once each feature is centred on its own mean)")
    print("exercise 1 looks right — every feature centred, every layer on one scale")

In [ ]:
_try("exercise 1", _check_prepare)

## 4. Exercise 2 — `concatenated_pca()`

The baseline every integration method is compared with, and the one most analysts try first: put the
prepared layers side by side, as one wide table with one row per donor, and take the principal components.
Components are ordered by the variance they explain, and they are orthogonal by construction. With the
singular value decomposition `X = U S V^T` of that table, the donor scores of component `j` are column
`j` of `U`, scaled here by the square root of the number of donors so that each has unit variance, and the
share of the total sum of squares that component explains is `S_j^2 / sum(S^2)`.

Do **not** centre or scale inside this function. That is exercise 1's job, and a function that quietly
re-centres hides the very mistake the next demo is about. The sign of a component is arbitrary (flip a
column of `U` and the matching row of `V^T` and nothing changes), so never compare scores without taking
that into account.

<details><summary>💡 Hint 1 — what to think about</summary>

Which library call gives you the three matrices of the decomposition? What shape does each have if you ask
for the economy version, and which of them holds the scores?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Stack the layers along the feature axis, decompose, keep the first `k` columns of the left matrix and
multiply them by the square root of the donor count. For the shares, square the singular values and divide
by their total. Raise `ValueError` for an empty list, layers that are not 2-D, layers with different
numbers of donors, a value that is not finite, `k` below 1 or above the smaller of the donor count and the
total number of features, and an all-zero table.
</details>

In [ ]:
class PCA(NamedTuple):
    scores: np.ndarray       # (donors, k) component scores, each with unit variance when the layers are centred
    explained: np.ndarray    # (k,) the share of the total sum of squares each component explains


def concatenated_pca(views: list, k: int) -> PCA:
    """Principal components of the layers placed side by side. No centring, no scaling.

    Stack the layers along the feature axis and take the economy singular value decomposition. The
    scores are the first `k` columns of the left singular vectors times the square root of the number of
    donors; `explained[j]` is the squared singular value of component `j` over the sum of all of them.

    Raise `ValueError` for: an empty list; a layer that is not 2-D; different numbers of donors; any value
    that is not finite; `k` below 1 or above min(donors, total features); a table that is all zeros.

    Example:
        >>> a = np.array([[3.0, 0.0], [0.0, 1.0], [-3.0, 0.0], [0.0, -1.0]])
        >>> r = concatenated_pca([a], 1)
        >>> np.abs(r.scores[:, 0]).round(4).tolist(), r.explained.round(2).tolist()
        ([1.4142, 0.0, 1.4142, 0.0], [0.9])

    Returns:
        A `PCA` of `scores` (donors, k) and `explained` (k,).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_pca() -> None:
    a = np.array([[3.0, 0.0], [0.0, 1.0], [-3.0, 0.0], [0.0, -1.0]])
    r = concatenated_pca([a], 2)
    assert r.scores.shape == (4, 2) and r.explained.shape == (2,), "scores are (donors, k) and explained is (k,)"
    assert np.allclose(np.abs(r.scores[:, 0]), [math.sqrt(2.0), 0.0, math.sqrt(2.0), 0.0], atol=1e-9), (
        "component 1 should follow the first column: its scores are the left singular vector times the root of the "
        f"number of donors; got {np.round(r.scores[:, 0], 4).tolist()}")
    assert np.allclose(r.explained, [0.9, 0.1], atol=1e-9), (
        "explained is the squared singular value over the sum of ALL squared singular values: expected [0.9, 0.1], "
        f"got {np.round(r.explained, 4).tolist()}")
    rng = np.random.default_rng(5)
    x1, x2 = rng.standard_normal((9, 4)), rng.standard_normal((9, 3))
    x1, x2 = x1 - x1.mean(axis=0), x2 - x2.mean(axis=0)
    r = concatenated_pca([x1, x2], 3)
    assert np.allclose(r.scores.T @ r.scores / 9, np.eye(3), atol=1e-9), (
        "component scores of centred layers are uncorrelated with unit variance: scale the left singular vectors by "
        "the square root of the number of donors")
    shifted = concatenated_pca([x1 + 5.0, x2 + 5.0], 1)
    assert not np.allclose(np.abs(shifted.scores), np.abs(r.scores[:, :1]), atol=1e-3), (
        "no centring inside this function: layers with a constant added must give different components")
    for bad_views, bad_k in (([], 1), ([a], 0), ([a], 5), ([np.zeros((4, 2))], 1), ([np.ones(4)], 1),
                             ([a, np.ones((3, 2))], 1)):
        try:
            concatenated_pca(bad_views, bad_k)
        except ValueError:
            continue
        raise AssertionError("an empty list, k outside 1..min(donors, features), an all-zero table, a 1-D layer or "
                             "layers with different donors must raise ValueError")
    print("exercise 2 looks right — components from the layers side by side")

In [ ]:
_try("exercise 2", _check_pca)

## 5. Exercise 3 — `best_assignment()`

To score a factor against the answer key you first have to decide which recovered factor is which planted
one. The factors come out in no particular order and with no particular sign, so the rule is: take the
absolute correlation between every recovered factor and every planted factor (`abs_corr_matrix`, given
above), and give each planted factor a **different** recovered factor so that the *total* correlation is as
large as possible.

The word that matters is *total*. A greedy rule, "take the largest correlation first, remove its row and
column, repeat", is the obvious thing to write and it is wrong: the largest entry can force a poor partner
on a different factor. When the scores are `[[0.9, 0.8], [0.85, 0.1]]`, the greedy rule takes 0.9 and is
left with 0.1; swapping the two gives a better total. You may recover more factors than were planted, so the
matrix has at least as many rows (recovered) as columns (planted); a recovered factor may be left unmatched.
The matrices here are tiny, so trying every one-to-one choice is affordable.

<details><summary>💡 Hint 1 — what to think about</summary>

How many ways are there to pick, for each planted factor in turn, a different recovered factor? Is that
small enough to try them all when you have a handful of each?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Go through every ordered choice of distinct rows, one per column, add up the scores it picks, and keep the
best. Return the rows as a tuple with one entry per column, and the best total. If several choices tie for
best, any of them is accepted. Raise `ValueError` for a score that is not 2-D, has no columns, has fewer
rows than columns, or holds anything that is not finite.
</details>

In [ ]:
def best_assignment(score: np.ndarray) -> tuple:
    """Give each column of `score` a different row so that the total score is as large as possible.

    `score[i, j]` is how well recovered factor `i` matches planted factor `j`. Return `(rows, total)`: `rows[j]`
    is the row chosen for column `j` (all different), and `total` is the sum of `score[rows[j], j]`. If
    several choices tie for the best total, any is accepted. Do not use a greedy rule.

    Raise `ValueError` if `score` is not 2-D, has no columns, has fewer rows than columns, or holds a value
    that is not finite.

    Example:
        >>> best_assignment(np.array([[0.9, 0.8], [0.85, 0.1]]))
        ((1, 0), 1.65)

    Returns:
        `(rows, total)`: a tuple of Python ints, and a float.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_assignment() -> None:
    rows, total = best_assignment(np.array([[0.9, 0.8], [0.85, 0.1]]))
    assert tuple(rows) == (1, 0) and abs(total - 1.65) < 1e-12, (
        "the best total here is 0.85 + 0.8, not the greedy 0.9 + 0.1: maximise the SUM over one-to-one choices, "
        f"not each column in turn; got rows {rows} with total {total}")
    s = np.array([[0.9, 0.8, 0.1], [0.85, 0.1, 0.2], [0.2, 0.3, 0.6], [0.1, 0.2, 0.5]])
    rows, total = best_assignment(s)
    assert len(set(rows)) == 3 and all(0 <= r < 4 for r in rows), f"rows must be distinct valid row numbers, got {rows}"
    assert abs(total - sum(s[r, j] for j, r in enumerate(rows))) < 1e-12, "total must be the sum of the chosen scores"
    assert abs(total - 2.25) < 1e-9, (
        "this matrix has an optimum no greedy rule finds: the largest entry is not always in the best assignment")
    assert all(isinstance(r, int) for r in rows), "return Python ints for the rows"
    rows, total = best_assignment(np.array([[0.4], [0.9], [0.3]]))
    assert tuple(rows) == (1,) and abs(total - 0.9) < 1e-12, "with one column the best row is simply the largest entry"
    for bad in (np.ones(3), np.ones((2, 3)), np.ones((3, 0)), np.array([[1.0, np.nan], [0.5, 0.2]])):
        try:
            best_assignment(bad)
        except ValueError:
            continue
        raise AssertionError("a 1-D score, fewer rows than columns, no columns, or a NaN must raise ValueError")
    print("exercise 3 looks right — the best one-to-one matching, not the greedy one")

In [ ]:
_try("exercise 3", _check_assignment)

In [ ]:
def match_to_planted(fitted: np.ndarray, planted: np.ndarray | None = None) -> tuple:
    """Given to you. Match fitted columns to planted ones (your `best_assignment` on absolute correlations).

    Returns `(rows, corr)`: `rows[j]` is the fitted column matched to planted factor `j`, and `corr[j]` is
    the absolute correlation between them.
    """
    planted = COHORT.factors if planted is None else planted
    score = abs_corr_matrix(fitted, planted)
    rows, _ = best_assignment(score)
    return list(rows), np.array([score[r, j] for j, r in enumerate(rows)])


def active_columns(views: list, z: np.ndarray, n_min: int = 0) -> list:
    """Given to you. The factors a fit reports: those explaining at least `ACTIVE_R2` of some layer.

    Uses your `variance_explained`. With the default `n_min=0` this is the honest list, and its length is the
    number of active factors every table below prints. If fewer than `n_min` columns qualify, the `n_min`
    strongest are returned instead, which is only ever done to complete a matching against the planted factors
    (every planted factor needs a partner); the tables say when it happened.
    """
    best = variance_explained(views, z).max(axis=0)
    chosen = [int(j) for j in np.flatnonzero(best >= ACTIVE_R2)]
    if len(chosen) < n_min:
        chosen = sorted(int(j) for j in np.argsort(-best, kind="stable")[:n_min])
    return chosen


PLANTED_PRESENT = np.array([(w != 0).any(axis=0) for w in COHORT.weights])    # (layers, factors): True where planted


def cells_right(views: list, z_matched: np.ndarray) -> int:
    """Given to you. Of the (layer, factor) cells, how many does a method call correctly present or absent?

    `z_matched[:, j]` is the method's factor matched to planted factor `j`; a cell is called present when
    the factor explains at least `ACTIVE_R2` of that layer. Compared with what was planted.
    """
    called = variance_explained(views, z_matched) >= ACTIVE_R2
    return int((called == PLANTED_PRESENT).sum())

## 6. Exercise 4 — `variance_explained()`

"How much of a layer does this factor explain?" is the question a factor model is read by, and it is also
how you can tell a shared factor from a layer-specific one. MOFA+ answers it with a coefficient of determination
for every factor in every data modality, calculated once the model is trained, from the model's own fitted
weights and factors (`claims.yaml`). This lesson computes the same kind of quantity with one difference: it
**refits the weights by least squares**, one factor at a time, instead of reading them from a model. Principal
components have no fitted weights of their own, so this is what lets principal components and model factors be
judged by one rule. Here it is, for one layer `Y` (donors x features, already centred) and one factor `z`:

- regress every feature (column of `Y`) on `z` alone, through the origin: the weight of feature `j` is
  `(Y[:, j] . z) / (z . z)`;
- the factor's R² in this layer is `1 - ||Y - z w^T||^2 / ||Y||^2`, which equals
  `sum_j (Y[:, j] . z)^2 / (z . z) / ||Y||^2`.

Least squares gives the smallest residual that any weights can give for a factor on its own, so this R² is at
least as large as the one the model's own weights (which its priors shrink) would give for the same factor. Read
it as an upper bound on the number a MOFA variance plot would show for that factor, not as that number. A factor
with no variation (sum of squares at most `2.0 ** -40`, about 9.1e-13, a power of two so that the edge is exact in
binary) explains 0. The numbers for different factors need not add up to the layer's total R²,
because factors need not be orthogonal: that is a property of the measure, not an error.

<details><summary>💡 Hint 1 — what to think about</summary>

You need the inner product of every feature with the factor. Can one matrix product give you all of them for
every factor at once? What do you divide by, and how do you avoid dividing by zero for a dead factor?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

For each layer, form `Y^T Z`: entry `(j, k)` is feature `j` with factor `k`. Square it, sum over features,
divide by the factor's sum of squares and by the layer's total sum of squares. Do that per layer and stack
the rows. Raise `ValueError` if `z` is not 2-D, a layer is not 2-D or has a different number of donors than
`z`, or a layer is all zeros.
</details>

In [ ]:
def variance_explained(views: list, z: np.ndarray) -> np.ndarray:
    """R^2 of each factor alone in each layer, with least-squares weights and no intercept.

    `views` are centred layers (donors x features); `z` is (donors, K). Entry `[m, k]` is
    `sum_j (Y_m[:, j] . z_k)^2 / (z_k . z_k) / ||Y_m||^2`. A factor whose sum of squares is at most
    `2.0 ** -40` (about 9.1e-13) explains 0 everywhere.

    Raise `ValueError` if `z` is not 2-D; a layer is not 2-D or its donors do not match `z`; or a layer is
    all zeros.

    Example:
        >>> y = np.array([[2.0, 0.0], [0.0, 0.0], [-2.0, 0.0], [0.0, 0.0]])
        >>> z = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0], [0.0, -1.0]])
        >>> variance_explained([y], z).round(3).tolist()
        [[1.0, 0.0]]

    Returns:
        A float array of shape (layers, K).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_variance_explained() -> None:
    y = np.array([[2.0, 0.0], [0.0, 0.0], [-2.0, 0.0], [0.0, 0.0]])
    z = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0], [0.0, -1.0]])
    got = variance_explained([y], z)
    assert got.shape == (1, 2) and np.allclose(got, [[1.0, 0.0]], atol=1e-12), (
        "a factor that reproduces the layer exactly explains all of it, and one that is orthogonal to every "
        f"feature explains nothing: expected [[1, 0]], got {np.round(got, 4).tolist()}")
    rng = np.random.default_rng(2)
    yy = rng.standard_normal((20, 5))
    yy -= yy.mean(axis=0)
    zz = rng.standard_normal((20, 3))
    got = variance_explained([yy, 2.0 * yy[:, :2]], zz)
    assert got.shape == (2, 3), "the result is (layers, factors)"
    for k in range(3):
        w = yy.T @ zz[:, k] / (zz[:, k] @ zz[:, k])
        want = 1.0 - ((yy - np.outer(zz[:, k], w)) ** 2).sum() / (yy ** 2).sum()
        assert abs(got[0, k] - want) < 1e-10, (
            "R^2 is 1 - ||Y - z w^T||^2 / ||Y||^2 with the least-squares weights w = Y^T z / (z . z); "
            f"factor {k}: expected {want:.4f}, got {got[0, k]:.4f}")
    assert np.allclose(got[1], variance_explained([2.0 * yy[:, :2]], zz)[0]) and np.allclose(
        variance_explained([3.0 * yy], zz), variance_explained([yy], zz)), "R^2 must not depend on the scale of the layer"
    assert np.allclose(variance_explained([yy], 5.0 * zz), variance_explained([yy], zz)), (
        "R^2 must not depend on the scale of the factor")
    dead = np.zeros((20, 1))
    assert variance_explained([yy], dead)[0, 0] == 0.0, "a factor with no variation explains 0, not NaN"
    for bad_views, bad_z in (([yy], zz[:, 0]), ([yy], zz[:10]), ([np.zeros((20, 2))], zz), ([yy[:, 0]], zz)):
        try:
            variance_explained(bad_views, bad_z)
        except ValueError:
            continue
        raise AssertionError("a 1-D z, mismatched donors, an all-zero layer or a 1-D layer must raise ValueError")
    print("exercise 4 looks right — R² per factor and per layer")

In [ ]:
_try("exercise 4", _check_variance_explained)

## 7. The baseline, measured

Everything below is computed. Two questions, asked of concatenated PCA given **the true number of
factors** (the model in the next sections will not be told it). First: does it matter that the layers were
put on one scale? Second: how do the PCA components compare with the planted factors, factor by factor,
and how much of each layer does each matched component explain, against how much the planted factor
explains?

In [ ]:
def _prepared() -> list:
    return prepare_views(COHORT.views)


def _pca_scales() -> None:
    n_true = len(FACTOR_NAMES)
    prepared = _prepared()
    raw = [v - v.mean(axis=0) for v in COHORT.views]
    out = {}
    for label, layers in (("centred only", raw), ("prepared", prepared)):
        r = concatenated_pca(layers, n_true)
        rows, corr = match_to_planted(r.scores)
        out[label] = (r, rows, corr)
    _PIPE["pca"] = out["prepared"]
    print(f"concatenated PCA given the true number of factors ({n_true}), matched one-to-one to the planted factors")
    print(f"  {'planted factor':16s} {'centred only':>13s} {'prepared':>9s}")
    for j, name in enumerate(FACTOR_NAMES):
        print(f"  {name:16s} {out['centred only'][2][j]:13.3f} {out['prepared'][2][j]:9.3f}")
    print(f"  {'mean':16s} {out['centred only'][2].mean():13.3f} {out['prepared'][2].mean():9.3f}")
    for label in out:
        print(f"  share of the total sum of squares explained by the first {n_true} components, {label}: "
              f"{100 * out[label][0].explained.sum():.1f}%")
    lone = FACTOR_NAMES.index("secretion")
    delta = out["prepared"][2][lone] - out["centred only"][2][lone]
    print(f"  the protein-only factor ({FACTOR_NAMES[lone]}) is recovered {'better' if delta > 0 else 'worse'} after "
          f"preparation by {abs(delta):.3f} in correlation")


_demo("PCA with and without preparation", _pca_scales, needs=_UPTO[3])

In [ ]:
def _variance_table() -> None:
    prepared = _prepared()
    r, rows, corr = _PIPE["pca"]
    planted = variance_explained(prepared, COHORT.factors)
    matched = variance_explained(prepared, r.scores[:, rows])
    print("variance explained, alone, by the planted factor and by the PCA component matched to it (percent of the layer)")
    print(f"  {'planted factor':16s} {'protein: planted':>17s} {'PCA':>6s}   {'RNA: planted':>13s} {'PCA':>6s}")
    for j, name in enumerate(FACTOR_NAMES):
        print(f"  {name:16s} {100 * planted[0, j]:17.1f} {100 * matched[0, j]:6.1f}   "
              f"{100 * planted[1, j]:13.1f} {100 * matched[1, j]:6.1f}")
    print(f"cells (layer x factor) called present or absent correctly at a threshold of {100 * ACTIVE_R2:g}%: "
          f"{cells_right(prepared, r.scores[:, rows])} of {PLANTED_PRESENT.size} for PCA")
    stress = FACTOR_NAMES.index("stress response")
    print(f"the component matched to {FACTOR_NAMES[stress]}, which was planted with no protein weights at all, explains "
          f"{100 * matched[0, stress]:.1f}% of the protein layer")


_demo("PCA variance explained", _variance_table, needs=_UPTO[4])

Read the first table for two things: whether the preparation moved each correlation, row by row and not only
the mean, and which planted factors PCA recovers least well. The tie is defined on the prepared scale, so it
exists only after preparation: on the raw scales the layers' different spreads break it, which is worth
having in mind when a row gets worse after preparation. Then read the second table for what the components say
about *where* a factor lives. A component is a mixture of whatever the data cannot separate. Where two planted
factors have the same total strength nothing in the variance of the data can choose between mixtures of them,
so the mixture PCA reports is set by the sampling noise of this one cohort (section 18 varies the strength to
test exactly that). A method that cannot tell you in which layer a component lives cannot tell you what to
measure next. The next sections build the model that can, and then test it on exactly these numbers.

## 8. The model, and the one number every update lowers

For each layer `m` (donors x features), the model says `Y_m ≈ Z W_m^T` plus noise. `Z` is shared: one row per
donor, one column per factor. `W_m` is private to the layer: one row per feature. Three kinds of unknown
are estimated:

- the factors `Z`, with a unit-variance prior (a ridge penalty pulling every donor's factor values towards 0);
- the weights of factor `k` in layer `m`, with a prior variance `a_mk` that this lesson calls the **scale**
  (a ridge penalty of its own for each factor in each layer);
- the noise: one variance `s_m` for every entry of layer `m`.

`J`, which `objective` (given above) computes, is minus the logarithm of the joint probability of the data and of
all these unknowns (the noise variances and scales included, with flat priors), up to a constant. You never need
its formula to write an update, because every update below is the **exact minimiser** of `J` over one block of
unknowns, with the others (and the floor `FLOOR`) held fixed. That is why `J` can only go down from sweep to
sweep, and you will check that it does. The cell below prints `J` for three principal components; a caution
about what `J` is not follows it.

- **Weights**, layer by layer: ridge regression of every feature on the factors. Factor `k` is penalised by
  `s_m / a_mk`, so a factor with a small scale in a layer is shrunk hard there. (Exercise 5.)
- **Factors**: ridge regression of every donor's features, *both layers together*, on the weights, each layer
  weighted by its noise precision `1 / s_m`, so the less noisy layer counts for more. (Exercise 6.)
- **Noise and scales**: `s_m` is the mean squared residual of layer `m`; `a_mk` is the mean squared weight of
  factor `k` in layer `m`, never below a floor. (Exercise 7.)

The scale update is what makes this a MOFA-style model rather than a ridge-regression factorisation. MOFA gives
every factor in every layer its own prior precision, learned from the data, and describes the effect as view- and
factor-wise sparsity: it identifies which factor is active in which layer (`claims.yaml`). Here, a factor whose
weights in a layer are small gets a small scale, so the next ridge step shrinks them further, and `J` rewards it
for it, through the `p_m / 2 log a_mk` term. A factor that has no work to do in a layer is switched off in that
layer, and a factor off in every layer is switched off for good.

Exercise 8 puts the steps in a loop. For the first `ARD_FROM` sweeps every scale is held at 1, so the sweeps are
ordinary alternating ridge regressions that find the part of the data the factors span; after that the scales
adapt. MOFA's first paper describes a similar burn-in, during which its sparsity constraints are deactivated to
avoid early splitting of factors (`claims.yaml`). Whether the warm-up matters for this model is a question
section 14 asks with a row of its own, and what happens without the adapting part is the rest of that section.

In [ ]:
def _objective_parts() -> None:
    prepared = _prepared()
    z = init_factors(prepared, 3)
    w = [np.linalg.lstsq(z, y, rcond=None)[0].T for y in prepared]    # least-squares weights: not your ridge update yet
    noise = [float(((y - z @ wm.T) ** 2).mean()) for y, wm in zip(prepared, w)]
    flat = [np.ones(3), np.ones(3)]
    by_data = [np.maximum((wm ** 2).mean(axis=0), FLOOR) for wm in w]
    print("J for three principal components with least-squares weights, with the scales held at 1, and then set from the weights:")
    print(f"  every scale at 1:                    J = {objective(prepared, z, w, noise, flat):.1f}")
    print(f"  each scale its mean squared weight:  J = {objective(prepared, z, w, noise, by_data):.1f}")


_demo("the objective and its scales", _objective_parts)

**A caution about `J`.** It has **no lowest value**. When a factor's weights in a layer are 0, the term
`p_m / 2 log a_mk` keeps falling as that scale `a_mk` falls, and only the floor `FLOOR` stops it. So a fit is
not "the minimum of `J`": it is the point that block descent reaches from its start, with every update exact over
its own block. Section 14 puts the `J` of a model with everything switched off next to the fit's at several floors,
and which one is lower depends on the floor. So `J` can compare fits that share the data, the number of factors
and the floor, and nothing more.

## 9. Exercise 5 — `update_weights()`

Given the factors `Z` (donors x K), one prepared layer `Y` (donors x features), the layer's noise variance `s`
and its scale for each factor `a` (a vector of length K), the weights that minimise `J` solve the ridge normal
equations. In matrix form, with `D = diag(1 / a)`:

`(Z^T Z + s D) W^T = Z^T Y`, so `W^T = (Z^T Z + s D)^-1 Z^T Y` and `W` has one row per feature.

That is a ridge regression of every feature on all the factors at once, with factor `k` penalised by `s / a_k`.
Solve the linear system directly instead of forming the inverse and multiplying.

<details><summary>💡 Hint 1 — what to think about</summary>

What is the shape of the matrix you must invert, and what does the penalty add to it? Which of the two
quantities, the noise variance or the scale, goes in the numerator?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Build the K-by-K matrix `Z^T Z` plus the noise variance times the diagonal matrix of `1 / scale`. Solve it
against `Z^T Y`, which is K by features, and transpose the answer so that features are the rows. Raise
`ValueError` for a noise variance that is not above 0, a scale vector that does not have one entry per
factor or has an entry not above 0, or a layer whose number of donors differs from the factors'.
</details>

In [ ]:
def update_weights(y: np.ndarray, z: np.ndarray, noise_var: float, scale2: np.ndarray) -> np.ndarray:
    """Weights of every factor in one layer: ridge regression with penalty `noise_var / scale2[k]` for factor k.

    Solve `(Z^T Z + noise_var * diag(1 / scale2)) W^T = Z^T Y` and return `W` with one row per feature.

    Raise `ValueError` if `noise_var` is not above 0, `scale2` is not a vector with one entry per factor and
    all above 0, or `y` and `z` do not have the same number of donors.

    Example:
        >>> z = np.array([[1.0], [-1.0], [2.0]])
        >>> y = np.array([[2.0, 1.0], [-2.0, -1.0], [4.0, 2.0]])
        >>> update_weights(y, z, 1.0, np.array([1.0])).round(4).tolist()
        [[1.7143], [0.8571]]

    Returns:
        An array of shape (features, K).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_update_weights() -> None:
    rng = np.random.default_rng(11)
    z = rng.standard_normal((30, 3))
    y = z @ rng.standard_normal((3, 4)) + 0.1 * rng.standard_normal((30, 4))
    s2 = np.array([0.5, 2.0, 1.0])
    w = update_weights(y, z, 0.8, s2)
    assert w.shape == (4, 3), f"the weights have one row per feature and one column per factor: expected (4, 3), got {w.shape}"
    resid = (z.T @ z + 0.8 * np.diag(1.0 / s2)) @ w.T - z.T @ y
    assert np.abs(resid).max() < 1e-9, (
        "the weights must solve (Z^T Z + noise_var * diag(1 / scale2)) W^T = Z^T Y: the penalty on factor k is "
        "noise_var / scale2[k], so a SMALL scale shrinks that factor's weights harder")
    loose = update_weights(y, z, 1e-9, np.ones(3))
    assert np.allclose(loose, np.linalg.lstsq(z, y, rcond=None)[0].T, atol=1e-6), "with almost no penalty this is ordinary least squares"
    tight = update_weights(y, z, 1.0, np.array([1e-9, 1.0, 1.0]))
    assert np.abs(tight[:, 0]).max() < 1e-4 and np.abs(tight[:, 1]).max() > 1e-2, (
        "a factor with a tiny scale must get weights close to 0 and the others must not: check the direction of the "
        "ratio (noise_var over scale2, not scale2 over noise_var)")
    for args in ((y, z, 0.0, s2), (y, z, 1.0, np.array([1.0, 1.0])), (y, z, 1.0, np.array([1.0, 0.0, 1.0])), (y[:10], z, 1.0, s2)):
        try:
            update_weights(*args)
        except ValueError:
            continue
        raise AssertionError("a noise variance of 0, a scale vector of the wrong length or with a 0, or mismatched donors "
                             "must raise ValueError")
    print("exercise 5 looks right — a ridge regression of every feature on the factors")

In [ ]:
_try("exercise 5", _check_update_weights)

## 10. Exercise 6 — `update_factors()`

Now fix the weights and noise variances and solve for the donors. Each donor's factor vector is the ridge
regression of that donor's features, over **both** layers, on the weights, with the layers weighted by their
noise precision (the less noisy a layer, the more it counts) and the unit-variance prior adding the identity
matrix. For all donors at once, with layers `m = 1..M`, weights `W_m` and noise variances `s_m`:

`Z = ( sum_m Y_m W_m / s_m ) ( I + sum_m W_m^T W_m / s_m )^-1`

This one step is where the layers are *integrated*: a factor that has weights in both layers is estimated
from both, and one that has weights only in one layer is estimated from that layer alone.

<details><summary>💡 Hint 1 — what to think about</summary>

Which two quantities are accumulated over the layers, and what shape does each have? What does the identity
matrix in the second one represent, and does it carry a noise variance?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Start a K-by-K matrix from the identity and a donors-by-K matrix from zeros. For each layer add the layer's
`W^T W` divided by its noise variance to the first and `Y W` divided by its noise variance to the second.
Solve the first against the transpose of the second and transpose back. Raise `ValueError` if the lists have
different lengths or are empty, a noise variance is not above 0, the weight matrices have different numbers
of factors, or a layer's features or donors do not match its weights or the other layers.
</details>

In [ ]:
def update_factors(views: list, weights: list, noise_vars) -> np.ndarray:
    """Donor factors given the weights: a noise-weighted ridge regression pooled over all layers.

    `Z = (sum_m Y_m W_m / s_m) (I + sum_m W_m^T W_m / s_m)^-1`, where `s_m` is `noise_vars[m]`.

    Raise `ValueError` if the three inputs are empty or have different lengths; a noise variance is not above
    0; the weight matrices do not all have the same number of columns; or `Y_m` does not have as many columns as
    `W_m` has rows, or the layers do not share their number of donors.

    Example:
        >>> y = [np.array([[1.0], [-1.0]]), np.array([[2.0], [-2.0]])]
        >>> w = [np.array([[1.0]]), np.array([[1.0]])]
        >>> update_factors(y, w, [1.0, 1.0]).round(4).tolist()
        [[1.0], [-1.0]]

    Returns:
        An array of shape (donors, K).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_update_factors() -> None:
    rng = np.random.default_rng(12)
    w = [rng.standard_normal((5, 2)), rng.standard_normal((7, 2))]
    y = [rng.standard_normal((9, 5)), rng.standard_normal((9, 7))]
    s = [0.6, 1.7]
    z = update_factors(y, w, s)
    assert z.shape == (9, 2), f"one row per donor and one column per factor: expected (9, 2), got {z.shape}"
    lhs = (np.eye(2) + w[0].T @ w[0] / s[0] + w[1].T @ w[1] / s[1]) @ z.T
    rhs = w[0].T @ y[0].T / s[0] + w[1].T @ y[1].T / s[1]
    assert np.abs(lhs - rhs).max() < 1e-9, (
        "each donor must solve (I + sum_m W_m^T W_m / s_m) z = sum_m W_m^T y_m / s_m: pool BOTH layers, weight each by "
        "its noise precision 1 / s_m, and keep the identity from the unit-variance prior")
    ignored = update_factors(y, w, [0.6, 1e12])
    only_first = update_factors(y[:1], w[:1], [0.6])
    assert np.allclose(ignored, only_first, atol=1e-6), "a layer with enormous noise variance must hardly count: weight by 1 / s_m"
    for args in (([], [], []), (y, w[:1], s), (y, w, [0.6, 0.0]), (y, [w[0], rng.standard_normal((7, 3))], s),
                 ([y[0], y[1][:, :6]], w, s), ([y[0], y[1][:5]], w, s)):
        try:
            update_factors(*args)
        except ValueError:
            continue
        raise AssertionError("empty input, lists of different lengths, a noise variance of 0, different numbers of factors "
                             "or mismatched features or donors must raise ValueError")
    print("exercise 6 looks right — the layers pooled, each weighted by its noise precision")

In [ ]:
_try("exercise 6", _check_update_factors)

## 11. Exercise 7 — `update_scales()`

Two quantities are re-estimated from what the other updates produced, for one layer at a time.

- **The noise variance** `s`: the mean squared residual of the layer, `mean((Y - Z W^T)^2)` over every
  entry. It is a variance, so it must stay above 0 for the next update to divide by it: take the larger of
  that mean and `floor`.
- **The scales** `a_k`, one per factor: the mean squared weight of factor `k` in this layer, the average over
  features of `W[:, k]^2`, not a variance around the column's mean, and not a sum. A factor with weights
  close to 0 in this layer gets a scale close to 0, which the next weight update punishes with a heavy penalty.
  Take the larger of that mean and `floor`, so that a factor can be very nearly off but its penalty
  stays finite.

The function returns both, in that order. The loop in the next exercise decides *when* the scales are allowed
to change; this function just computes what they would become.

<details><summary>💡 Hint 1 — what to think about</summary>

Which axis do you average the squared weights over, features or factors? Which array does the residual use,
the weights this layer has now or something else?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Residual is the layer minus factors times the transposed weights; average its square over everything. Square
the weights, average down the features so you get one number per factor, and raise both results to the floor
with an element-wise maximum.
</details>

In [ ]:
def update_scales(y: np.ndarray, z: np.ndarray, w: np.ndarray, floor: float = FLOOR) -> tuple:
    """New noise variance and scales for one layer, from the current factors and weights.

    `noise_var = max(mean((y - z @ w.T) ** 2), floor)` over all entries; `scale2[k] = max(mean(w[:, k] ** 2),
    floor)`, the mean over the features of the squared weights of factor k.

    Raise `ValueError` if `floor` is not above 0 or the shapes do not fit (`y` (n, p), `z` (n, K), `w` (p, K)).

    Example:
        >>> z = np.array([[1.0], [-1.0]])
        >>> w = np.array([[2.0], [0.0]])
        >>> y = np.array([[2.0, 1.0], [-2.0, -1.0]])
        >>> noise_var, scale2 = update_scales(y, z, w)
        >>> round(float(noise_var), 4), scale2.round(4).tolist()
        (0.5, [2.0])

    Returns:
        `(noise_var, scale2)`: a float and an array of shape (K,).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_update_scales() -> None:
    z = np.array([[1.0], [-1.0]])
    w = np.array([[2.0], [0.0]])
    y = np.array([[2.0, 1.0], [-2.0, -1.0]])
    s, a = update_scales(y, z, w)
    assert abs(s - 0.5) < 1e-12 and a.shape == (1,) and abs(a[0] - 2.0) < 1e-12, (
        "the noise variance is the mean squared residual of the whole layer, and the scale is the mean over features of the "
        f"squared weights of each factor: expected (0.5, [2.0]), got ({s}, {a.tolist()})")
    rng = np.random.default_rng(13)
    z, w, y = rng.standard_normal((10, 3)), rng.standard_normal((6, 3)), rng.standard_normal((10, 6))
    w[:, 1] = 0.0
    s, a = update_scales(y, z, w, 1e-3)
    assert a.shape == (3,) and abs(a[1] - 1e-3) < 1e-15, "a factor with all-zero weights gets the floor, not 0 and not NaN"
    assert abs(a[0] - float((w[:, 0] ** 2).mean())) < 1e-12, (
        "average the squared weights down the features: no centring (not a variance), and not a sum")
    assert abs(s - float(((y - z @ w.T) ** 2).mean())) < 1e-12, "the residual uses the current factors and weights"
    s, _ = update_scales(z @ w.T, z, w, 1e-6)
    assert s == 1e-6, "a perfect fit must still leave a noise variance above 0: apply the floor"
    for args in ((y, z, w, 0.0), (y, z[:5], w, 1e-6), (y, z, w[:, :2], 1e-6)):
        try:
            update_scales(*args)
        except ValueError:
            continue
        raise AssertionError("a floor of 0 or shapes that do not fit must raise ValueError")
    print("exercise 7 looks right — noise variance and one scale per factor")

In [ ]:
_try("exercise 7", _check_update_scales)

## 12. Exercise 8 — `fit_factor_model()`

Put the three updates in a loop. A **sweep** is one pass, in this order:

1. for each layer, new weights from `update_weights`, using the current factors, noise variance and scales;
2. new factors from `update_factors`, using those new weights and the current noise variances;
3. for each layer, `update_scales` on the new factors and the weights from step 1: its noise variance always
   replaces the layer's, and its scales replace the layer's scales **only when the sweep number (counting
   from 0) is at least `ard_from`**.

Start from the given factors `z0`, every noise variance at 1 and every scale at 1. The first `ard_from`
sweeps therefore hold the scales at 1; they are ordinary alternating ridge regressions. Each step is an
exact minimiser of `J`, so `J` after any number of sweeps is no higher than after one fewer, up to rounding.
The check below uses exactly that, and it is the quickest way to find a wrong update.

<details><summary>💡 Hint 1 — what to think about</summary>

What state does the loop carry from one sweep to the next? Which of the three steps uses a value produced by
an earlier step in the same sweep, and which uses one left over from the previous sweep?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Keep the factors, a list of weight matrices, an array of noise variances and a layers-by-factors array of scales.
In each sweep fill the weights layer by layer, replace the factors, then ask `update_scales` about each
layer. Overwrite the noise variance every sweep, and overwrite the scale row only from sweep `ard_from`. Raise
`ValueError` for fewer than 1 sweep, a negative `ard_from`, a `floor` not above 0, an empty list of layers, or
a `z0` that is not 2-D or does not have one row per donor.
</details>

In [ ]:
class FactorModel(NamedTuple):
    z: np.ndarray            # (donors, K) the factors
    weights: list            # one (features, K) array per layer
    noise_var: np.ndarray    # (layers,) the noise variance of each layer
    scale2: np.ndarray       # (layers, K) the scale of each factor in each layer


def fit_factor_model(views: list, z0: np.ndarray, n_sweeps: int, ard_from: int = ARD_FROM,
                     floor: float = FLOOR) -> FactorModel:
    """Fit the multi-view factor model by alternating the three updates for `n_sweeps` sweeps.

    State: the factors (start `z0`, copied), the noise variance of each layer (start 1) and the scale of each
    factor in each layer (start 1). One sweep: (1) weights of each layer with `update_weights`; (2) the factors
    with `update_factors`, using the new weights; (3) `update_scales` for each layer on the new factors and the
    weights from (1): its noise variance always replaces the layer's, its scales replace the layer's only when
    the sweep index (from 0) is at least `ard_from`. `floor` is passed on to `update_scales`.

    Raise `ValueError` if `n_sweeps` is below 1, `ard_from` is negative, `floor` is not above 0, `views` is empty,
    or `z0` is not 2-D with one row per donor.

    Example:
        >>> y = [np.array([[1.0, 2.0], [-1.0, -2.0], [2.0, 4.0], [-2.0, -4.0]])]
        >>> m = fit_factor_model(y, np.array([[1.0], [-1.0], [1.0], [-1.0]]), 3, ard_from=0)
        >>> m.z.shape, [w.shape for w in m.weights], m.noise_var.shape, m.scale2.shape
        ((4, 1), [(2, 1)], (1,), (1, 1))

    Returns:
        A `FactorModel`: `z`, `weights`, `noise_var` and `scale2`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_fit() -> None:
    cohort = generate_cohort(21, n_donors=40)
    views = prepare_views([cohort.views[0][:, :30], cohort.views[1][:, :60]])
    z0 = init_factors(views, 3, seed=4)
    kept = z0.copy()
    m = fit_factor_model(views, z0, 6, ard_from=2)
    assert np.array_equal(z0, kept), "do not change z0 in place: copy it"
    assert m.z.shape == (40, 3) and [w.shape for w in m.weights] == [(30, 3), (60, 3)], "factors (donors, K), one weight matrix per layer"
    assert m.noise_var.shape == (2,) and m.scale2.shape == (2, 3), "noise_var is (layers,) and scale2 is (layers, K)"
    values = []
    for sweeps in (1, 2, 3, 5, 8, 13, 21):
        f = fit_factor_model(views, z0, sweeps, ard_from=2)
        values.append(objective(views, f.z, f.weights, f.noise_var, f.scale2))
    worst = max(b - a for a, b in zip(values, values[1:]))
    assert worst <= 1e-9 * max(abs(v) for v in values), (
        "the objective must never rise from one sweep to the next: every update is an exact minimiser of it. Check the order "
        "(weights, then factors, then noise and scales), that update_scales sees the NEW factors, and that the noise "
        f"variance is updated every sweep; it rose by {worst:.3g}")
    held = fit_factor_model(views, z0, 2, ard_from=2)
    assert np.all(held.scale2 == 1.0), "with ard_from = 2 and 2 sweeps (indices 0 and 1) no scale may have changed yet"
    moved = fit_factor_model(views, z0, 3, ard_from=2)
    assert np.any(moved.scale2 != 1.0), "the sweep with index ard_from must already update the scales"
    assert np.all(held.noise_var != 1.0), "the noise variance is updated from the very first sweep, whatever ard_from is"
    for kwargs in (dict(n_sweeps=0), dict(n_sweeps=3, ard_from=-1), dict(n_sweeps=3, floor=0.0)):
        try:
            fit_factor_model(views, z0, **{"n_sweeps": 3, **kwargs})
        except ValueError:
            continue
        raise AssertionError("n_sweeps below 1, a negative ard_from or a floor of 0 must raise ValueError")
    print("exercise 8 looks right — the objective falls on every sweep, and the scales wait for ard_from")

In [ ]:
_try("exercise 8", _check_fit)

## 13. The model on the cohort, checked against the planted factors

Now run it on the cohort, started from PCA (the starting point MOFA+'s own software adopted, `claims.yaml`) with
**more factors than the cohort has**, so the model has to decide how many to keep. MOFA's first paper did the
same: it started from 25 factors and pruned them during training with a minimum fraction of variance explained
that the user specifies, 2% in that paper. As of 14 October 2026 the MOFA2 manual pages give different defaults:
15 initial factors, and a drop-factor threshold of -1, which means no dropping (`claims.yaml`). This lesson's fit
never prunes; the scale update does the switching off, and a factor is called *active* afterwards, by the rule
`ACTIVE_R2`. The first cell checks the property that justifies the design: the objective never rises. The second
compares what the model recovered with what PCA recovered and with what was planted.

In [ ]:
def _fit_main() -> None:
    prepared = _prepared()
    z0 = init_factors(prepared, K_FIT)
    checkpoints = (1, 2, 4, 8, 16, 32, 64, 128, 256, N_SWEEPS)
    values = []
    for sweeps in checkpoints:
        f = fit_factor_model(prepared, z0, sweeps)
        values.append(objective(prepared, f.z, f.weights, f.noise_var, f.scale2))
        if sweeps == N_SWEEPS:
            _PIPE["fit"] = f
    print("objective J after k sweeps, from the PCA start: " + ", ".join(f"{k}: {v:.1f}" for k, v in zip(checkpoints, values)))
    rose = [k for k, a, b in zip(checkpoints[1:], values, values[1:]) if b > a + 1e-9 * abs(a)]
    print(f"J rose between checkpoints: {rose if rose else 'never'}")


_demo("objective on the cohort", _fit_main, needs=_UPTO[8])

In [ ]:
def _model_table() -> None:
    prepared = _prepared()
    f = _PIPE["fit"]
    pca, pca_rows, pca_corr = _PIPE["pca"]
    n_true = len(FACTOR_NAMES)
    active = active_columns(prepared, f.z)                  # what the fit reports: no padding
    pool = active_columns(prepared, f.z, n_true)            # the same, completed only if a planted factor would have no partner
    rows, corr = match_to_planted(f.z[:, pool])
    cols = [pool[r] for r in rows]
    _PIPE["model_cols"] = cols
    planted_r2 = variance_explained(prepared, COHORT.factors)
    model_r2 = variance_explained(prepared, f.z[:, cols])
    pca_r2 = variance_explained(prepared, pca.scores[:, pca_rows])
    print(f"the model started with {K_FIT} factors and reports {len(active)} as active "
          f"(explaining at least {100 * ACTIVE_R2:g}% of some layer); the cohort has {n_true}"
          + (f". {len(pool) - len(active)} inactive column(s) were added to the matching only, so that every planted factor has a partner"
             if len(pool) > len(active) else ""))
    print(f"  {'planted factor':16s} {'|corr| PCA':>10s} {'model':>6s} | {'protein R2%: planted':>20s} {'PCA':>5s} {'model':>6s} | "
          f"{'RNA R2%: planted':>16s} {'PCA':>5s} {'model':>6s}")
    for j, name in enumerate(FACTOR_NAMES):
        print(f"  {name:16s} {pca_corr[j]:10.3f} {corr[j]:6.3f} | {100 * planted_r2[0, j]:20.1f} {100 * pca_r2[0, j]:5.1f} "
              f"{100 * model_r2[0, j]:6.1f} | {100 * planted_r2[1, j]:16.1f} {100 * pca_r2[1, j]:5.1f} {100 * model_r2[1, j]:6.1f}")
    print(f"  {'mean':16s} {pca_corr.mean():10.3f} {corr.mean():6.3f}")
    print(f"cells (layer x factor) called present or absent correctly: PCA {cells_right(prepared, pca.scores[:, pca_rows])}, "
          f"model {cells_right(prepared, f.z[:, cols])}, of {PLANTED_PRESENT.size}")
    stress = FACTOR_NAMES.index("stress response")
    a = f.scale2[:, cols[stress]]
    print(f"the model's scale for {FACTOR_NAMES[stress]} is {a[0]:.2g} in the protein layer, where the factor was planted "
          f"with no weights, and {a[1]:.2g} in the RNA layer")
    planted_noise = [1.0 - float(((w ** 2).sum(axis=0) / RAW_SD[m] ** 2).sum() / w.shape[0]) for m, w in enumerate(COHORT.weights)]
    print("noise variance as a share of each layer: model "
          + ", ".join(f"{VIEW_NAMES[m]} {f.noise_var[m]:.3f}" for m in range(2)) + "; planted "
          + ", ".join(f"{VIEW_NAMES[m]} {planted_noise[m]:.3f}" for m in range(2)))


_demo("model against planted factors", _model_table, needs=_UPTO[8])

In [ ]:
def _plot_r2() -> None:
    prepared = _prepared()
    f = _PIPE["fit"]
    pca, pca_rows, _ = _PIPE["pca"]
    tables = [("planted", variance_explained(prepared, COHORT.factors)),
              ("concatenated PCA", variance_explained(prepared, pca.scores[:, pca_rows])),
              ("factor model", variance_explained(prepared, f.z[:, _PIPE["model_cols"]]))]
    top = max(float(t.max()) for _, t in tables)
    fig, axes = plt.subplots(1, 3, figsize=(9, 2.9), sharey=True)
    for ax, (title, r2) in zip(axes, tables):
        ax.imshow(r2.T, vmin=0.0, vmax=top, cmap="viridis", aspect="auto")
        ax.set_title(title, fontsize=9)
        ax.set_xticks(range(len(VIEW_NAMES)))
        ax.set_xticklabels(VIEW_NAMES, fontsize=8)
        for j in range(len(FACTOR_NAMES)):
            for m in range(len(VIEW_NAMES)):
                ax.text(m, j, f"{100 * r2[m, j]:.0f}", ha="center", va="center", fontsize=8,
                        color="white" if r2[m, j] < 0.6 * top else "black")
    axes[0].set_yticks(range(len(FACTOR_NAMES)))
    axes[0].set_yticklabels(FACTOR_NAMES, fontsize=8)
    fig.suptitle("variance explained, alone, by the factor matched to each planted factor (percent of the layer)", fontsize=9)
    fig.tight_layout()
    _show(fig)


_demo("variance explained, drawn", _plot_r2, needs=_UPTO[8])

Read the table in three passes. First the correlations, factor by factor: where is the model's number above PCA's,
and by how much, and is there a factor where it is not? Second the R² columns: compare each recovered column with
the planted one in the layer where the factor was planted **and** in the layer where it was not. Third the last
lines: how many factors survived (and whether the matching had to borrow an inactive column), and what scale the
model gave the RNA-only factor in the protein layer. Do not read a verdict into one cohort: sections 18 and 19
repeat this on other worlds. What this table can already show is which of the quantities you wrote in exercises
4 to 8 are doing the work that PCA could not.

## 14. What the scales buy: take them away

Exercise 8's one unusual ingredient is the scale that adapts. To see what it does, run the same model with
the scales held at 1 for every sweep (`ard_from` beyond the last sweep, so the three updates are plain alternating
ridge regressions), and compare. Two starting points each, because the starting point is the other thing
that decides where an iteration ends up: the PCA start the main fit used, and a random start.

Two numbers describe each fit. The first is the one you have used: the absolute correlation of each planted
factor with the best single fitted factor. The second ignores how the fitted factors are mixed: how much of each
planted factor lies in the **span** of all the fitted factors together (`span_r2`, given above). If the two
disagree, the fit has found the right directions and mixed them up. A span is computed over every factor a fit
reports (at least as many columns as there are planted factors), and covering a planted factor with more columns
can only make the span larger, so the table prints how many columns each span used.

In [ ]:
def _score_fit(prepared: list, cohort: Cohort, z: np.ndarray) -> dict:
    """Score one set of fitted factors against the answer key.

    `active` is the number of factors the fit reports (explaining at least ACTIVE_R2 of some layer): the honest
    count, which can be below the number planted. `padded` is how many inactive columns were added so that every
    planted factor could still be matched to a column; it is 0 whenever the fit reports as many as were planted.
    """
    n_true = cohort.factors.shape[1]
    active = active_columns(prepared, z)
    pool = active_columns(prepared, z, n_true)
    rows, corr = match_to_planted(z[:, pool], cohort.factors)
    cols = [pool[r] for r in rows]
    return {"corr": corr, "cols": cols, "active": len(active), "padded": len(pool) - len(active),
            "cells": cells_right(prepared, z[:, cols]), "span": span_r2(z[:, pool], cohort.factors),
            "span_cols": len(pool)}


def _ablation() -> None:
    prepared = _prepared()
    z_pca = init_factors(prepared, K_FIT)
    z_rand = init_factors(prepared, K_FIT, seed=SEED + 1)
    runs = {
        "full model, PCA start": _PIPE["fit"],
        "scales adapt from sweep 0, PCA start": fit_factor_model(prepared, z_pca, N_SWEEPS, ard_from=0),
        "scales never adapt, PCA start": fit_factor_model(prepared, z_pca, N_SWEEPS, ard_from=N_SWEEPS),
        "full model, random start": fit_factor_model(prepared, z_rand, N_SWEEPS),
        "scales adapt from sweep 0, random start": fit_factor_model(prepared, z_rand, N_SWEEPS, ard_from=0),
        "scales never adapt, random start": fit_factor_model(prepared, z_rand, N_SWEEPS, ard_from=N_SWEEPS),
    }
    _PIPE["ablation"] = runs
    width = 42
    print(f"{'':{width}s} {'active':>6s} {'mean |corr|':>12s} {'min |corr|':>11s} {'span: mean':>11s} {'min':>6s} {'cols':>5s} {'cells right':>12s} {'J':>9s}")
    pca, pca_rows, pca_corr = _PIPE["pca"]
    span = span_r2(pca.scores, COHORT.factors)
    print(f"{'concatenated PCA (true number of factors)':{width}s} {pca.scores.shape[1]:6d} {pca_corr.mean():12.3f} {pca_corr.min():11.3f} "
          f"{span.mean():11.3f} {span.min():6.3f} {pca.scores.shape[1]:5d} {cells_right(prepared, pca.scores[:, pca_rows]):12d} {'':>9s}")
    borrowed = []
    for label, f in runs.items():
        s = _score_fit(prepared, COHORT, f.z)
        j = objective(prepared, f.z, f.weights, f.noise_var, f.scale2)
        if s["padded"]:
            borrowed.append(label)
        print(f"{label:{width}s} {s['active']:6d} {s['corr'].mean():12.3f} {s['corr'].min():11.3f} {s['span'].mean():11.3f} "
              f"{s['span'].min():6.3f} {s['span_cols']:5d} {s['cells']:12d} {j:9.1f}")
    print(f"rows with fewer active factors than were planted, whose matching had to borrow an inactive column: "
          f"{borrowed if borrowed else 'none'}")


_demo("scales taken away", _ablation, needs=_UPTO[8])

Four comparisons, in this order. First, the "span" columns across the model rows: is the subspace that the
planted factors live in found whether or not the scales adapt? Compare spans only between rows that used the
same number of columns (the "cols" column): a span over more columns is larger by construction, so a row with
more columns than the rest has a head start that the span alone does not show. Second, the correlation columns across the same
rows: does it matter whether the scales adapt, and does it matter where the fit starts? Third, the "active"
column: how many factors does each fit report (the line under the table says when a row's matching had to borrow
an inactive column)? Fourth, the rows that adapt the scales from the very first sweep, against the full-model
rows above them: they ask whether the warm-up of `ARD_FROM` sweeps buys anything on this world.

A fit whose span is right and whose single-factor correlations are poor has recovered the right directions and
mixed them, which is the rotation problem of section 1 made visible. The last column is the same function `J`
in every row. The rows whose scales never adapt did not minimise it over the scales (they hold every scale at
1), so read their `J` as the value at a point that was not optimised over them; if two such rows print the same
`J`, it is because with every scale at 1 the function does not change when the factors are rotated. Compare `J`
between fits only when both were allowed to minimise over the same unknowns, with the same floor. The next
cell shows what the floor does to it.

In [ ]:
FLOORS = (FLOOR, 1e-9, 1e-12)


def _floor_demo() -> None:
    prepared = _prepared()
    n = prepared[0].shape[0]
    z0 = init_factors(prepared, K_FIT)
    print("J of the fit, and of an empty model: no factor has any weight in any layer, every scale is at the floor,")
    print("every noise variance is the layer's mean square (the fit is refitted from the PCA start at each floor)")
    print(f"  {'floor':>7s} | {'J of the fit':>13s} | {'J of the empty model':>21s} | {'lower':>15s} | {'fit: mean |corr|':>16s}")
    for floor in FLOORS:
        f = _PIPE["fit"] if floor == FLOOR else fit_factor_model(prepared, z0, N_SWEEPS, floor=floor)
        j_fit = objective(prepared, f.z, f.weights, f.noise_var, f.scale2)
        j_empty = objective(prepared, np.zeros((n, K_FIT)), [np.zeros((v.shape[1], K_FIT)) for v in prepared],
                            [float((v ** 2).mean()) for v in prepared], np.full((len(prepared), K_FIT), floor))
        corr = _score_fit(prepared, COHORT, f.z)["corr"]
        print(f"  {floor:7.0e} | {j_fit:13.1f} | {j_empty:21.1f} | {'the fit' if j_fit < j_empty else 'the empty model':>15s} | {corr.mean():16.3f}")


_demo("J without its floor", _floor_demo, needs=_UPTO[8])

Read the "lower" column against the correlation column. If the empty model has the lower `J` at some floor, then
`J` is lower at a point that explains nothing than at a fit that recovers the planted factors (if the last column
says it does), and nothing in `J` alone tells the two apart. What keeps a fit away from the empty model is block
descent from a sensible start, not a minimum of `J`. That is also why this lesson uses `J` only to compare fits
that share the data, the number of factors, the start rule and the floor, and never as a score of how right a fit
is.

## 15. How many sweeps?

The scales do not settle in one step: a factor drifts towards one layer over several sweeps. So the number of
sweeps is a result, not a setting to copy. The cell below fits the model from two starting points with a growing number of
sweeps and scores each fit against the planted factors. The scales are held at 1 for the first `ARD_FROM` sweeps
(the warm-up of section 8), so every row below has to run longer than the warm-up before a scale has moved at
all; the cell prints how many sweeps of adaptation each row got. MOFA's software caps its iterations too, at a
default that the documentation states (`claims.yaml`), and judges convergence by the ELBO.

In [ ]:
SWEEP_STEPS = (50, 100, 400, 1600)


def _sweeps() -> None:
    prepared = _prepared()
    starts = {"PCA start": init_factors(prepared, K_FIT), "random start": init_factors(prepared, K_FIT, seed=SEED + 1)}
    print(f"{'sweeps':>7s} {'adapting':>9s} | " + " | ".join(f"{label + ': mean |corr|, active, J':>38s}" for label in starts))
    table = {}
    for sweeps in SWEEP_STEPS:
        cells = []
        for label, z0 in starts.items():
            f = fit_factor_model(prepared, z0, sweeps)
            s = _score_fit(prepared, COHORT, f.z)
            table[(sweeps, label)] = s["corr"].mean()
            cells.append(f"{s['corr'].mean():14.3f} {s['active']:8d} {objective(prepared, f.z, f.weights, f.noise_var, f.scale2):13.1f}")
        print(f"{sweeps:7d} {max(sweeps - ARD_FROM, 0):9d} | " + " | ".join(cells))
    _PIPE["sweeps"] = table
    gain = {label: table[(SWEEP_STEPS[-1], label)] - table[(SWEEP_STEPS[0], label)] for label in starts}
    print(f"mean |corr| gained between {SWEEP_STEPS[0]} and {SWEEP_STEPS[-1]} sweeps: "
          + ", ".join(f"{label} {gain[label]:+.3f}" for label in starts))


_demo("how many sweeps", _sweeps, needs=_UPTO[8])

Read the "adapting" column first: it is the number of sweeps in which the scales were free to move, so the first
row is the fit whose scales had the fewest sweeps in which to move. Then read down each start: where does the
mean correlation stop changing, and does `J` stop changing at the same row? The two starts need not stop at the
same row, and the printed gain says how much is still left to gain by running longer than the first row.

## 16. Many random restarts, and what an objective can and cannot tell you

The objective is not convex, so where the iteration starts matters, and MOFA's authors say so: they ran many
random restarts and kept the one with the highest evidence lower bound (ELBO), and note that the optimisation
landscape is not convex, so a global optimum is not guaranteed (`claims.yaml`). The cell below runs `N_RESTARTS` random
starts on this cohort and prints, for every run, what you could know without an answer key (the objective `J`,
the data likelihood `data_nll`, the number of active factors) next to what you could not (the correlation with
the planted factors).

`data_nll` is the likelihood of the data with the factors integrated out, and it has no prior on the weights. It
cannot see how the factors are rotated, because an orthogonal mix of the factors does not change the covariance the
model gives the data. `J` has the per-layer scales in it, so it can. `data_nll` is **not** the bound MOFA compares
restarts by: that bound (the ELBO) belongs to MOFA's full probabilistic model, with priors on the factors and the
weights and an approximate posterior over them (`claims.yaml`), so it contains the sparsity prior that `data_nll`
leaves out and is closer in kind to `J`. What `data_nll` cannot see is therefore not a statement about MOFA's
bound, and this notebook does not measure whether that bound can see rotation.

The table prints each run's `data_nll` as the amount above the lowest of the runs (the likelihood itself is almost
the same for all of them, and a column of near-identical numbers would hide that some run is lower). `J` is
comparable across these runs because they share the data, the number of factors, the update rules and the floor of
section 14, and for no other reason.

In [ ]:
def _restarts() -> None:
    prepared = _prepared()
    fits = []
    for r in range(N_RESTARTS):
        fits.append(fit_factor_model(prepared, init_factors(prepared, K_FIT, seed=SEED + 100 + r), N_SWEEPS))
    _PIPE["restarts"] = fits
    rows = []
    for f in fits:
        s = _score_fit(prepared, COHORT, f.z)
        rows.append((objective(prepared, f.z, f.weights, f.noise_var, f.scale2), data_nll(prepared, f.weights, f.noise_var),
                     s["active"], float(s["corr"].mean()), float(s["corr"].min())))
    _PIPE["restart_rows"] = rows
    j, nll, act, mean_c, min_c = (np.array(col) for col in zip(*rows))
    print(f"{'run':>4s} {'J':>9s} {'data nll above lowest':>22s} {'active':>7s} {'mean |corr|':>12s} {'min |corr|':>11s}")
    for r in range(N_RESTARTS):
        print(f"{r:4d} {j[r]:9.1f} {nll[r] - nll.min():22.4f} {act[r]:7d} {mean_c[r]:12.3f} {min_c[r]:11.3f}")
    print(f"J ranges over {j.max() - j.min():.1f} (lowest {j.min():.1f}); data nll over {nll.max() - nll.min():.3f}; "
          f"mean |corr| from {mean_c.min():.3f} to {mean_c.max():.3f}")
    extra = [r for r in range(N_RESTARTS) if act[r] > len(FACTOR_NAMES)]
    print(f"runs reporting more active factors than the cohort has: {extra if extra else 'none'}"
          + (f"; their J minus the lowest J: {[round(float(j[r] - j.min()), 1) for r in extra]}" if extra else ""))
    best_j, best_nll, best_truth = int(np.argmin(j)), int(np.argmin(nll)), int(np.argmax(mean_c))
    print(f"the run with the lowest J is run {best_j} (mean |corr| {mean_c[best_j]:.3f}); with the lowest data nll, run {best_nll} "
          f"(mean |corr| {mean_c[best_nll]:.3f}); with the best correlation, which you would not know, run {best_truth} "
          f"({mean_c[best_truth]:.3f})")


_demo("random restarts", _restarts, needs=_UPTO[8], limit=240)

Read the columns you could know against the one you could not. How far apart are the runs in `J`, and how far in
the data likelihood? Is there a run with more active factors than the cohort has, or a `J` far above the rest, and
if there is, what does its correlation look like? Among the runs that share almost the same `J`, how far apart do
the correlations still sit? If they sit far apart, the objective is nearly flat along the directions that mix the
factors, which is the rotation problem seen from the other side. A restart has to be chosen by something, and the
something has to work without the answer key.

## 17. Exercise 9 — `consensus_run()`

One choice needs neither an objective nor an answer key: take the run that **agrees most with the others**. If
the restarts mostly find the same factors, the run in the middle of that agreement is a fair representative, and a
run that disagrees with all the others is the one to doubt. It is the rule to reach for when objectives cannot be
compared (runs from different software, or from models with different priors); it does not promise a better pick
than the lowest `J` when `J` is comparable, and the cell after this exercise prints both picks side by side.
MOFA's software offers the same idea as a heat map of factor correlations between runs, which should look
block-diagonal when factors are found robustly (`claims.yaml`).

`runs` is a list of arrays, one per restart, each with one row per donor and one column per *active* factor of that
run (the runs may have different numbers of columns). For a pair of runs, match the columns of the one with
fewer columns to the columns of the other with `best_assignment` on `abs_corr_matrix`, and call their agreement
the mean matched absolute correlation, the total divided by the smaller number of columns. Fill an `R x R`
matrix with the agreement of every pair (1.0 on the diagonal). A run's score is the mean of its agreements with the
**other** runs, and the consensus run is the one with the highest score; among runs within 1e-9 of the highest
score, the one with the lowest index.

<details><summary>💡 Hint 1 — what to think about</summary>

Which of your earlier functions already turns two sets of factors into "how well do they match"? Which of the
two runs plays the role of the planted factors, when their numbers of columns differ? Is the diagonal part of a
run's score?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

For every pair `i < j`, put the run with fewer columns second (the first if they have the same number), build the
absolute correlation matrix between the larger and the smaller, match them, divide the total by the smaller number
of columns, and store it in both `[i, j]` and `[j, i]`. Score each run as the row sum without the diagonal over
`R - 1`. Return the index and the matrix. Raise `ValueError` for fewer than two runs, a run that is not 2-D or has no
columns, or runs that do not share their number of donors.
</details>

In [ ]:
def consensus_run(runs: list) -> tuple:
    """The restart that agrees most with the others, and the matrix of pairwise agreements.

    `agreement[i, j]` is the mean absolute correlation of the optimal one-to-one matching between the columns of
    the run with fewer columns and the columns of the other (`best_assignment` on `abs_corr_matrix`); the
    diagonal is 1.0. A run's score is the mean of its agreements with the other runs. Return the index of the highest
    score, taking the lowest index among scores within 1e-9 of the highest, and the agreement matrix.

    Raise `ValueError` for fewer than 2 runs, a run that is not 2-D or has no columns, or runs with different
    numbers of donors.

    Example:
        >>> a = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0], [0.0, -1.0]])
        >>> best, agreement = consensus_run([a, a[:, ::-1], np.array([[1.0, 1.0], [1.0, 2.0], [3.0, 5.0], [1.0, 9.0]])])
        >>> best, agreement.round(2).tolist()[0]
        (0, [1.0, 1.0, 0.81])

    Returns:
        `(best, agreement)`: an int and an (R, R) float array.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_consensus() -> None:
    a = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0], [0.0, -1.0]])
    odd = np.array([[1.0, 1.0], [1.0, 2.0], [3.0, 5.0], [1.0, 9.0]])
    best, agree = consensus_run([odd, a, a[:, ::-1] * -1.0])
    assert best == 1, (
        "the two runs that contain the same factors (in another order, with a flipped sign) agree perfectly, so the consensus is "
        f"one of them, the lower index; got run {best}. Use ABSOLUTE correlations and match columns one-to-one")
    assert agree.shape == (3, 3) and np.allclose(agree, agree.T) and np.allclose(np.diag(agree), 1.0), "agreement is symmetric with 1.0 on the diagonal"
    assert abs(agree[1, 2] - 1.0) < 1e-9, "a run and a reordered, sign-flipped copy of it must agree at 1.0"
    best, _ = consensus_run([a, a])
    assert best == 0, "when scores tie, take the lowest index"
    wide = np.hstack([a, odd[:, :1]])
    _, agree = consensus_run([a, wide])
    assert abs(agree[0, 1] - 1.0) < 1e-9, (
        "with different numbers of columns, match the smaller set into the larger one and divide by the SMALLER number of columns")
    for bad in ([a], [a, a[:3]], [a, a[:, :0]], [a, np.ones(4)]):
        try:
            consensus_run(bad)
        except ValueError:
            continue
        raise AssertionError("fewer than two runs, runs with different donors, a run without columns or a 1-D run must raise ValueError")
    print("exercise 9 looks right — the restart that agrees most with the others")

In [ ]:
_try("exercise 9", _check_consensus)

In [ ]:
def _consensus_demo() -> None:
    prepared = _prepared()
    fits = _PIPE["restarts"]
    rows = _PIPE["restart_rows"]
    runs = [f.z[:, active_columns(prepared, f.z, 1)] for f in fits]
    best, agreement = consensus_run(runs)
    others = (agreement.sum(axis=1) - 1.0) / (len(runs) - 1)
    j = np.array([r[0] for r in rows])
    mean_c = np.array([r[3] for r in rows])
    print("mean agreement of each run with the others: " + ", ".join(f"{r}: {others[r]:.3f}" for r in range(len(runs))))
    for label, pick in (("the consensus run", best), ("the run with the lowest J", int(np.argmin(j)))):
        rank = int((mean_c > mean_c[pick]).sum()) + 1           # ties share a rank
        print(f"{label} is run {pick}; its mean |corr| with the planted factors is {mean_c[pick]:.3f}, "
              f"which ranks {rank} of {len(runs)} runs")
    print(f"for reference, the median run is {float(np.median(mean_c)):.3f} and the best run is {mean_c.max():.3f}")
    low = int(np.argmin(others))
    print(f"the run that agrees least with the others is run {low}; its mean |corr| is {mean_c[low]:.3f}")


_demo("the consensus run", _consensus_demo, needs=_UPTO[9])

Compare the two picks. Which run has the higher mean |corr| with the planted factors, the one `J` chose or the one
the consensus chose, and by how much against the spread of all the runs? Both rules saw the same runs and
neither saw the key. `J` contains the per-layer scales, so unlike the data likelihood it can tell rotated versions
of one fit apart (section 16); the consensus run uses only the factors, so it needs no objective at all. The
runs of one cohort cannot rank two rules, and this lesson does not repeat the restarts over the worlds of
section 19, which would multiply that section's cost by the number of restarts. What the numbers above license is
narrower: if the pick of `J` ranks at least as high as the consensus pick, nothing here recommends the consensus
run over `J` when `J` is comparable. What the consensus adds is a choice that does not depend on an objective, and
an agreement matrix that shows a restart disagreeing with all the others.

## 18. From a tie to well-separated factors

The planted tie is the hard case for PCA, on purpose. Real factors are rarely tied exactly, so the fair question is
how fast the difficulty goes away as one factor gets weaker than the other. `generate_cohort` takes a `gap`:
the strength of proliferation divided by the strength of stress response. Here the gap grows from a tie, and for
each gap the cell builds `WORLDS_PER_GAP` cohorts and compares concatenated PCA, given the true number of
factors, with the model started from PCA. The cohorts come from the **same seeds at every gap**: between two rows
of the table only the strength of stress response changes, so the rows are paired, and a difference between them
is not world-to-world luck. The score for each cohort is the worse of the two correlations, for proliferation and
for stress response, since a method that recovers one and loses the other has not separated them.

In [ ]:
IDX_PAIR = (FACTOR_NAMES.index("proliferation"), FACTOR_NAMES.index("stress response"))


def _analyse_world(seed: int, gap: float = 1.0) -> dict:
    """Prepare a cohort, run concatenated PCA and the model from the PCA start, and score both against the key."""
    cohort = generate_cohort(seed, gap)
    prepared = prepare_views(cohort.views)
    n_true = cohort.factors.shape[1]
    pca = concatenated_pca(prepared, n_true)
    pca_rows, pca_corr = match_to_planted(pca.scores, cohort.factors)
    fit = fit_factor_model(prepared, init_factors(prepared, K_FIT), N_SWEEPS)
    s = _score_fit(prepared, cohort, fit.z)
    return {"pca": pca_corr, "model": s["corr"], "pca_cells": cells_right(prepared, pca.scores[:, pca_rows]),
            "model_cells": s["cells"], "active": s["active"], "padded": s["padded"]}


def _gap_sweep() -> None:
    table = {}
    print(f"{'gap':>5s} | {'PCA: worse of the pair, mean [min-max]':>40s} | {'model':>26s} | {'model above PCA':>16s} | {'active factors':>14s}")
    for gap in GAPS:
        worlds = [_analyse_world(SEED + 1000 + r, gap) for r in range(WORLDS_PER_GAP)]
        table[gap] = worlds
        pca = np.array([min(w["pca"][list(IDX_PAIR)]) for w in worlds])
        mod = np.array([min(w["model"][list(IDX_PAIR)]) for w in worlds])
        print(f"{gap:5.2f} | {pca.mean():10.3f} [{pca.min():.3f}-{pca.max():.3f}]{'':>9s} | {mod.mean():6.3f} [{mod.min():.3f}-{mod.max():.3f}] | "
              f"{int((mod > pca).sum()):>10d} of {WORLDS_PER_GAP} | {', '.join(str(w['active']) for w in worlds):>14s}")
    _PIPE["gaps"] = table
    short = sum(1 for worlds in table.values() for w in worlds if w["padded"])
    print(f"cohorts, over all rows, with fewer active factors than were planted (their matching borrowed an inactive column): "
          f"{short} of {len(GAPS) * WORLDS_PER_GAP}")
    first, last = GAPS[0], GAPS[-1]
    gain = [np.mean([min(w["model"][list(IDX_PAIR)]) - min(w["pca"][list(IDX_PAIR)]) for w in table[g]]) for g in (first, last)]
    print(f"mean advantage of the model over PCA on the worse of the pair: {gain[0]:+.3f} at gap {first:g}, {gain[1]:+.3f} at gap {last:g}")


_demo("tie to separated", _gap_sweep, needs=_UPTO[8], limit=300)

In [ ]:
def _plot_gap() -> None:
    table = _PIPE["gaps"]
    fig, ax = plt.subplots(figsize=(6, 3.2))
    for label, key, colour in (("concatenated PCA", "pca", "tab:orange"), ("factor model", "model", "tab:blue")):
        mean = [np.mean([min(w[key][list(IDX_PAIR)]) for w in table[g]]) for g in GAPS]
        low = [min(min(w[key][list(IDX_PAIR)]) for w in table[g]) for g in GAPS]
        high = [max(min(w[key][list(IDX_PAIR)]) for w in table[g]) for g in GAPS]
        ax.errorbar(GAPS, mean, yerr=[np.array(mean) - np.array(low), np.array(high) - np.array(mean)], marker="o", capsize=3,
                    label=label, color=colour)
    ax.set_xlabel("gap: strength of proliferation over strength of stress response (1 is a tie)")
    ax.set_ylabel("worse of the pair, |corr|")
    ax.legend(fontsize=8)
    fig.tight_layout()
    _show(fig)


_demo("tie to separated, drawn", _plot_gap, needs=_UPTO[8])

Read the table along the gap. The first row is the tie; the last has the weaker factor at a fraction of the
stronger one's strength. Two things to look for: how the PCA column changes as the factors separate, and whether
the model's advantage changes with it, including whether it keeps one at the end. PCA can only separate the two
factors once they differ in variance; the model has another handle, which layers each one lives in. The "active
factors" column lists, per cohort, how many factors the model reports: a cohort with fewer than the planted
number has at least one planted factor with no active column to match, and the line under the table counts them.

## 19. How much of that was the luck of one seed, and of the strength of the factors?

Everything in sections 7 to 17 is one cohort. The first cell below builds `N_WORLDS` cohorts from consecutive
seeds, at the tie, and runs the comparison on each. It prints the range across worlds, not an average alone: a
mean that hides a world in which a method fails would be the wrong thing to quote. The second cell changes
something else: the strength of the planted factors.

In [ ]:
def _spread() -> None:
    worlds = [_analyse_world(SEED + r) for r in range(N_WORLDS)]
    _PIPE["spread"] = worlds
    print(f"|corr| with the planted factor over {N_WORLDS} worlds: mean [min-max]")
    print(f"  {'planted factor':16s} {'PCA':>22s} {'model':>22s}")
    for j, name in enumerate(FACTOR_NAMES):
        p = np.array([w["pca"][j] for w in worlds])
        m = np.array([w["model"][j] for w in worlds])
        print(f"  {name:16s} {p.mean():8.3f} [{p.min():.3f}-{p.max():.3f}] {m.mean():8.3f} [{m.min():.3f}-{m.max():.3f}]")
    pc = np.array([w["pca_cells"] for w in worlds])
    mc = np.array([w["model_cells"] for w in worlds])
    print(f"cells (layer x factor) called right, of {PLANTED_PRESENT.size}: PCA mean {pc.mean():.1f} [{pc.min()}-{pc.max()}], "
          f"model mean {mc.mean():.1f} [{mc.min()}-{mc.max()}]")
    print("active factors reported by the model, world by world: " + ", ".join(str(w["active"]) for w in worlds)
          + f" (planted: {len(FACTOR_NAMES)})")
    short = [r + 1 for r, w in enumerate(worlds) if w["padded"]]   # numbered from 1, in the order of the line above
    print(f"worlds with fewer active factors than were planted (their matching borrowed an inactive column; numbered "
          f"from 1 as above): {', '.join(map(str, short)) if short else 'none'}")
    worse = [r + 1 for r, w in enumerate(worlds) if w["model"].mean() < w["pca"].mean()]
    print(f"worlds in which the model's mean |corr| is below PCA's (numbered from 1 as above): "
          f"{', '.join(map(str, worse)) if worse else 'none'}")
    print(f"the first of these worlds is the cohort studied in sections 2 to 17 (seed {SEED}), so the fit you already saw is one of the {N_WORLDS}")


_demo("seed spread", _spread, needs=_UPTO[8], limit=300)

Read "min" before "mean". The promise a method makes is about the worlds you have not seen, so the worst world
in the table matters more than the average one. Then read the last lines: the number of active factors, world
by world, and the worlds in which the model does worse than PCA, if there are any. A method that is better on
average and worse in some worlds is a different statement from one that is better in all of them, and the table
does not decide which one the data in your own lab will be.

Every world above had the factors at the strength of section 2. The next cell makes them weaker: it multiplies
every planted share of variance by a number below 1 (`WEAK_SHARES`) and uses fewer donors (`WEAK_DONORS`), in
`WEAK_WORLDS` worlds each, and for every world prints the planted factor that each method recovers least well.
Two rows to look for: a world in which the model's worst factor is far below PCA's worst, and a world in which
the model reports fewer active factors than were planted. The model is fitted exactly as before. Two things
make the comparison uneven, and the cell prints what it needs to read them. PCA is handed the true number of
factors in every cell of this section, whereas the model starts from more and has to decide how many to keep.
And a planted factor whose own share of every layer falls under `ACTIVE_R2` would not count as active however
well it was recovered, so for every world the cell also prints how many of the planted factors are active
*as planted*, and the share of its best layer that the weakest planted factor explains, next to the model's count.

What to expect from the scale update you wrote: it sets a factor's scale from the size of that factor's own
weights and nothing else. A weak factor has small weights, so it gets a small scale, so the next ridge step
shrinks them further, and nothing in the update asks how uncertain they are. If that costs something, it shows as
a very low worst correlation together with a model count below the planted-as-active count.

In [ ]:
def _weak_worlds() -> None:
    n_true = len(FACTOR_NAMES)
    print(f"{WEAK_WORLDS} worlds of {WEAK_DONORS} donors per setting (seeds {SEED} onwards). Worst = the planted factor a method recovers least well")
    for scale in WEAK_SHARES:
        pca_worst, model_worst, pca_name, model_name, active, key_active, key_weakest = [], [], [], [], [], [], []
        for r in range(WEAK_WORLDS):
            cohort = generate_cohort(SEED + r, 1.0, WEAK_DONORS, scale)
            prepared = prepare_views(cohort.views)
            _, pca_corr = match_to_planted(concatenated_pca(prepared, n_true).scores, cohort.factors)
            fit = fit_factor_model(prepared, init_factors(prepared, K_FIT), N_SWEEPS)
            s = _score_fit(prepared, cohort, fit.z)
            pca_worst.append(float(pca_corr.min()))
            model_worst.append(float(s["corr"].min()))
            pca_name.append(FACTOR_NAMES[int(np.argmin(pca_corr))])
            model_name.append(FACTOR_NAMES[int(np.argmin(s["corr"]))])
            active.append(s["active"])
            key_active.append(len(active_columns(prepared, cohort.factors)))
            key_weakest.append(100.0 * float(variance_explained(prepared, cohort.factors).max(axis=0).min()))
        print(f"planted shares x{scale:g}:")
        print(f"  worst |corr| per world, PCA:   " + "  ".join(f"{v:.2f}" for v in pca_worst) + f"   ({', '.join(pca_name)})")
        print(f"  worst |corr| per world, model: " + "  ".join(f"{v:.2f}" for v in model_worst) + f"   ({', '.join(model_name)})")
        print(f"  active factors reported by the model, per world: " + ", ".join(str(a) for a in active) + f" (planted: {n_true})")
        print(f"  planted factors that are active as planted, per world:   " + ", ".join(str(a) for a in key_active)
              + f" (threshold {100 * ACTIVE_R2:g}% of some layer; the weakest planted factor's best layer, % explained: "
              + ", ".join(f"{v:.1f}" for v in key_weakest) + ")")
        below = [r + 1 for r in range(WEAK_WORLDS) if model_worst[r] < pca_worst[r]]   # numbered from 1, left to right
        print(f"  worlds in which the model's worst factor is recovered less well than PCA's worst (numbered from 1, left to "
              f"right): {', '.join(map(str, below)) if below else 'none'}")


_demo("weaker factors", _weak_worlds, needs=_UPTO[8], limit=300)

Read each pair of rows, then the list of worlds in which the model's worst factor is below PCA's. Read the
model's count of active factors against the count of planted factors that are active *as planted*, not against the
number planted: where the planted count is itself lower, the world put a factor under the threshold, and where the
model's count is lower, the model switched off a factor that the data contained. The weakest planted factor's share
is measured on the cohort, not the generator's nominal one (on a few donors, least squares on many noisy features
tends to read a little more than the nominal share out of the noise). If a world loses a factor, the advantage
measured in sections 18 and 19 holds at the strengths of section 2 and not beyond them. MOFA+ is fitted by
variational inference, which approximates a distribution for each unknown instead of one value (`claims.yaml`);
whether that changes this result is not measured here.

## 20. What this model is, and what it is not

You have built the idea at the centre of MOFA, at the size of a laptop. It is worth being exact about how it differs
from the software it is named after, because every difference below is a place where a real analysis can behave
differently from this notebook. What follows about MOFA, MOFA+ and totalVI is as of 14 October 2026; `claims.yaml`
records each source and the date it was read.

- **Inference.** You fit point estimates by alternating exact minimisations of a function that has no lowest
  value without its floor (section 14). MOFA+ is fitted by variational inference, which approximates a
  distribution for each unknown (`claims.yaml`). With random starts, MOFA's papers compare restarts by a bound
  on the evidence of the whole model (section 16); MOFA+'s software starts from principal components instead
  (`claims.yaml`). Section 19 looks at one place where point estimates may cost something: weak factors.
- **Noise.** One variance per layer here. MOFA's Gaussian noise model gives every feature its own precision
  (`claims.yaml`). The cohort here has equal noise within a layer by construction, so the assumption was
  free; in real data it is not.
- **Sparsity.** One scale per factor and layer, which is the factor-wise, layer-wise sparsity. MOFA adds a
  second level, feature-wise sparsity, so that a factor touches a small number of features
  (`claims.yaml`).
- **What it can read.** Complete matched layers on a Gaussian scale. MOFA ignores missing values in its likelihood
  rather than imputing them, and has other likelihoods for binary and count data (`claims.yaml`); this model has
  none of that, and its input should already be normalised.
- **Technical factors.** Here a batch is planted and the model is left to find it, so that you can measure
  how much of each layer it carries. MOFA's documentation recommends regressing clear technical factors out
  *before* fitting, because otherwise the model spends its factors on them (`claims.yaml`).

**The half that is not here.** The catalogue's lab also runs totalVI on CITE-seq data. totalVI represents the data as a
composite of biological and technical factors, including protein background and batch effects (`claims.yaml`).
Its generative process uses neural networks, its documentation lists, among its limitations, that it effectively
requires a GPU for fast inference, and it ships in scvi-tools, which depends on PyTorch (`claims.yaml`). MOFA+'s own
Python package declares pandas, scipy, scikit-learn and h5py as requirements (`claims.yaml`). None of those is in
this course's environment. So this lesson does not compare factor models with deep generative models; it builds the
factor-model half and measures it against a known answer, which is the part a learner can verify. The cell below
measures one of the assumptions above, on this cohort: how much the residual variance of single features varies
within a layer, where the noise is equal by construction, next to what equal noise alone would give.

In [ ]:
def _noise_check() -> None:
    prepared = _prepared()
    f = _PIPE["fit"]
    kept = f.z[:, active_columns(prepared, f.z)]
    rng = np.random.default_rng(SEED + 2000)
    print("residual variance of each feature as a multiple of its layer's mean: 5th percentile, median, 95th percentile")
    print(f"  {'':8s} {'after the fit':>16s}   {'equal noise, same donors and factors':>40s}")
    for m, y in enumerate(prepared):
        v = ((y - f.z @ f.weights[m].T) ** 2).mean(axis=0)
        lo, mid, hi = np.percentile(v / v.mean(), [5, 50, 95])
        noise = rng.standard_normal(y.shape)
        noise -= kept @ np.linalg.solve(kept.T @ kept, kept.T @ noise)
        ref = (noise ** 2).mean(axis=0)
        rlo, rmid, rhi = np.percentile(ref / ref.mean(), [5, 50, 95])
        print(f"  {VIEW_NAMES[m]:8s} {lo:6.2f}, {mid:.2f}, {hi:.2f}   {rlo:15.2f}, {rmid:.2f}, {rhi:.2f}")


_demo("noise within a layer", _noise_check, needs=_UPTO[8])

Compare the two bands layer by layer. A spread cannot be read on its own: each feature's residual variance is
estimated from a limited number of donors, so it varies from feature to feature even when the noise is exactly
equal. The right-hand band is that spread for pure noise with one variance in every feature, for the same donors,
after the same fitted factors were taken out by least squares. If the band after the fit is no wider than it, the
spread you see is sampling, and nothing in this cohort contradicts the assumption of equal noise (which holds here
by construction). A band clearly wider than the reference would mean that some features are noisier than others;
real data has that, and one noise variance per layer would then be wrong for many of its features.

## 21. Common mistakes

- **Comparing factors without a matching, or with a greedy one.** Factors come out in any order and with any
  sign. Section 5's function is why the tables above can be read at all.
- **Reading a correlation as "found it".** A correlation with the best single factor is one number; the span
  overlap in section 14 says whether the right directions were found and mixed.
- **Pooling layers that are on different scales.** Section 7's first table compares the layers as they stand
  with the layers prepared. Read it row by row: the tie is defined on the prepared scale, so raw scales break it.
- **Trusting one restart, or one seed.** Sections 15 to 19 each vary one thing the single fit in section 13 held fixed.
- **Choosing a restart by a likelihood that cannot see rotation.** Section 16 prints this lesson's `data_nll` next
  to the correlation, run by run. It is not MOFA's bound, which belongs to the full model with its priors.
- **Reading "active" as "real".** A factor above a threshold in a layer is a statement about this fit; the
  threshold is a choice (`ACTIVE_R2`), the planted shares of section 2 were set clear of it, and section 19
  shows what changes when they are not.
- **Reading a count that was padded.** A matching needs a partner for every planted factor, so a table that
  completed it with inactive columns must say so, and the counts in this notebook are the unpadded ones.
- **Treating `J` as a score.** It has no lowest value without its floor, so compare it only between fits that
  share the data, the start rule and the floor (section 14).
- **Treating factor shares as additive.** Each is computed for one factor alone.
- **Reporting a factor without saying how it was fitted.** The methods cell at the end is the minimum.

In [ ]:
def _methods_statement() -> None:
    prepared = _prepared()
    f = _PIPE["fit"]
    active = active_columns(prepared, f.z)
    print("METHODS — a factor model without these lines cannot be reproduced or compared")
    print(f"  data            {prepared[0].shape[0]} donors; layers: " + ", ".join(
        f"{VIEW_NAMES[m]} {v.shape[1]} features" for m, v in enumerate(prepared)))
    print("  preparation     every feature centred; every layer divided by the root of its mean square")
    print(f"  model           {K_FIT} factors to start; one noise variance per layer; one scale per factor and layer; "
          f"scales held at 1 for {ARD_FROM} sweeps, floor {FLOOR:g}")
    print(f"  fitting         {N_SWEEPS} sweeps of alternating exact minimisations; start from the first {K_FIT} principal components")
    print(f"  factors kept    those explaining at least {100 * ACTIVE_R2:g}% of some layer: {len(active)} here")
    print(f"  restarts        {N_RESTARTS} random starts compared with each other (section 17), alongside the PCA start; none chosen with the answer key")
    print("  limits          point estimates; one noise variance per layer; complete data; Gaussian scale")


_demo("methods statement", _methods_statement, needs=_UPTO[9])

## 22. Self-check

1. Two planted factors have exactly the same total strength: one is shared by both layers, the other is present only in the RNA layer. What can concatenated PCA do with them?
   - (a) return one component for each of them, because principal components are orthogonal
   - (b) return two components that mix them, in a proportion set by the cohort's sampling noise, because equal variances leave the mixture undecided
   - (c) return the shared factor first and the RNA-only factor second, because a shared factor has more weights

2. In the fitted model a factor's weights in one layer are very small while its weights in the other layer are large. What does the model's scale for that factor in the first layer do?
   - (a) it grows, which loosens the penalty and lets those weights grow back
   - (b) it stays at 1, because only the noise variance is re-estimated
   - (c) it shrinks, so the ridge penalty on those weights gets heavier and shrinks them further

3. The data likelihood (factors integrated out) is almost the same for two fits whose factors correlate very differently with the planted ones. Why is that no contradiction?
   - (a) mixing the factors with an orthogonal matrix, and the weights back, leaves the covariance the fit implies unchanged, and the likelihood depends on the factors only through it
   - (b) both fits must have kept the same number of factors
   - (c) the likelihood is dominated by the noise variances, so it ignores the factors

4. Five factors explain 9%, 8%, 7%, 6% and 5% of a layer, each computed alone. Can you say that together they explain 35% of it?
   - (a) yes, shares of variance always add up
   - (b) no, because a factor's share is always smaller than the first factor's
   - (c) not necessarily: each share is for one factor alone, so correlated factors explain some of the same variance, and only uncorrelated factors' shares add up

5. The objective is lowest for one restart, but that restart disagrees with all the others about two factors. What is the sensible next step?
   - (a) report it: the objective is the only principled criterion
   - (b) look at what is different about it (how many factors it keeps, how its factors relate to the others), run more restarts, and compare with a PCA start before reporting either
   - (c) average the factors of all the restarts, since averaging always removes the disagreement

Answers, with the reasoning, come with this lesson's worked solution when you enrol on Synapsa.

## What you built, and where it goes next

You now have a measuring stick and a model. The stick is a matching of recovered factors to planted ones, a
per-layer variance-explained table and a stability check across restarts and seeds. The model is a few lines of
alternating ridge regressions with one idea added to them: a scale for every factor in every layer, which lets
the model say that a factor lives in one layer and not the other. The habit to carry forward is the one the
measuring stick enforces: **a recovered factor is a claim, and the claim has to survive a comparison with
something that could have contradicted it.** The next lesson in this programme builds a harness for the ways such
claims fail across layers (the same donor on both sides of a split, a batch confounded with the outcome), and the
capstone runs a factor model inside that harness.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_prepare),
                              ("exercise 2", _check_pca),
                              ("exercise 3", _check_assignment),
                              ("exercise 4", _check_variance_explained),
                              ("exercise 5", _check_update_weights),
                              ("exercise 6", _check_update_factors),
                              ("exercise 7", _check_update_scales),
                              ("exercise 8", _check_fit),
                              ("exercise 9", _check_consensus)):
            _try(_name, _check)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.